# Complete GRF study — all six tasks, resumable Colab execution

This version is configured for **research training**, not smoke tests. It uses the
model code that completed your T4 smoke tests, with an all-task controller and a
direct importer for your supplied **Viglialoro2025 external workbook**.

**Start:** select a GPU runtime, then **Runtime → Run all**. Authorize your Drive
mount and select `Shida-LOSO-ANKLE.zip` if no cached copy is present. The external
workbook is embedded below, so you do not need to upload it again.

The notebook sequentially runs:

1. Data/protocol checks for all six components/sides.
2. Subject-wise nested LOSO for all participants and all seven model families.
3. Absolute and translation-normalized input comparisons with the available ZIP.
4. Additional requested coordinate/perturbation analyses when their real inputs exist.
5. Full-internal-cohort selection/refitting and model freeze for the primary coordinates.
6. External domain-shift evaluation when its identity/units/visit mapping is complete.

**The full supplied-data plan has an upper bound of 206,322 model fits**, including
two coordinate variants and final selection/refits at ten deep candidates per
family. This is a large research run and may need many Colab sessions. Drive
checkpoints resume completed fits. Stopping the runtime does not continue training
in the background; reconnect, rerun this notebook, and keep the same experiment
name/settings. Do not run two sessions against the same experiment folder.

**Known external-data gap:** the uploaded workbook has 12 populated numeric sheets,
each 101 samples × 23 anonymous columns. It has no participant/trial/visit labels,
mass, or pelvis data; the six heel sheets are empty. The columns must not be
declared 23 independent participants. Vertical forces range from approximately
−1,312 to 0, so units and signs must be source-verified before N/kg evaluation.
The importer creates the missing-metadata template and explicitly blocks external
results until it is completed. Internal training continues independently.

**Validation:** your prior notebook completed both neural smoke tests on a T4 and
passed 23 protocol tests. This controller/importer is additionally tested locally.
No full research run or external evaluation is claimed by this notebook file.


## 1 Runtime and Google Drive

Use the same runtime package versions on resume. A dependency change is detected by the saved experiment configuration.


In [ ]:
#@title Install notebook dependencies and check TensorFlow
import os
import sys
import json
import re
import shutil
import hashlib
import subprocess
import importlib.util
import importlib.metadata
from pathlib import Path
from datetime import datetime, timezone

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Open this .ipynb file in Google Colab before running the setup cells.")
if sys.version_info < (3, 10):
    raise RuntimeError("A Python 3.10+ Colab runtime is required.")
if importlib.util.find_spec("tensorflow") is None:
    raise RuntimeError("TensorFlow is not installed. Select a standard Colab Python GPU runtime.")

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                "keras-tuner==1.4.7", "openpyxl==3.1.5"], check=True)
probe = r"""
import json, tensorflow as tf, keras_tuner, keras
print(json.dumps({"tensorflow": tf.__version__, "keras": keras.__version__,
                  "keras_tuner": keras_tuner.__version__,
                  "gpus": [d.name for d in tf.config.list_physical_devices('GPU')]}))
"""
process = subprocess.run([sys.executable, "-c", probe], text=True, capture_output=True)
if process.returncode:
    print(process.stdout)
    print(process.stderr)
    raise RuntimeError("TensorFlow/KerasTuner import failed. Reset the runtime and rerun this setup cell.")
RUNTIME_INFO = json.loads(process.stdout.strip().splitlines()[-1])
tf_version = tuple(int(v) for v in RUNTIME_INFO["tensorflow"].split(".")[:2])
if tf_version < (2, 16):
    raise RuntimeError("TensorFlow 2.16 or newer is required for the selected Keras backend.")
for package in ["numpy", "pandas", "openpyxl", "scikit-learn"]:
    RUNTIME_INFO[package] = importlib.metadata.version(package)
RUNTIME_INFO["python"] = sys.version
print(json.dumps(RUNTIME_INFO, indent=2))
if not RUNTIME_INFO["gpus"]:
    print("No GPU detected. The small smoke test can use CPU; GPU is recommended for study training.")


In [ ]:
#@title Project folder
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_PROJECT_FOLDER = "/content/drive/MyDrive/Shida_Nested_LOSO" #@param {type:"string"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_FOLDER).expanduser()
else:
    PROJECT_DIR = Path("/content/Shida_Nested_LOSO")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
CODE_DIR = Path("/content/shida_nested_loso_code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA_DIR = Path("/content/shida_runtime_data")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_module(relative_path, source):
    destination = CODE_DIR / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
    print(f"Ready: {relative_path}")

print("Project:", PROJECT_DIR)
print("Persistent across runtime resets:", USE_GOOGLE_DRIVE)


## 2 Internal data

The cached Drive copy from your previous notebook is reused. Otherwise upload the original ZIP; do not unzip it.


In [ ]:
#@title Dataset source
DATA_SOURCE = "upload" #@param ["upload", "drive"]
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Shida-LOSO-ANKLE.zip" #@param {type:"string"}
USE_CACHED_UPLOAD = True #@param {type:"boolean"}

from zipfile import ZipFile, is_zipfile
from google.colab import files

cached_dataset = PROJECT_DIR / "datasets" / "Shida-LOSO-ANKLE.zip"
if DATA_SOURCE == "drive":
    selected_zip = Path(DRIVE_ZIP_PATH).expanduser()
elif USE_CACHED_UPLOAD and cached_dataset.is_file():
    selected_zip = cached_dataset
    print("Using cached upload:", selected_zip)
else:
    upload_directory = LOCAL_DATA_DIR / "uploads"
    upload_directory.mkdir(parents=True, exist_ok=True)
    previous_directory = Path.cwd()
    try:
        os.chdir(upload_directory)
        uploaded = files.upload()
    finally:
        os.chdir(previous_directory)
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one dataset ZIP: Shida-LOSO-ANKLE.zip.")
    selected_zip = upload_directory / zip_names[0]
    del uploaded

if not selected_zip.is_file() or not is_zipfile(selected_zip):
    raise ValueError(f"A readable ZIP was not found: {selected_zip}")
with ZipFile(selected_zip) as archive:
    raw_files = [name for name in archive.namelist()
                 if "/raw/" in ("/" + name.lower()) and name.lower().endswith(".xlsx")]
    if not raw_files:
        raise ValueError("This ZIP has no Raw participant workbooks. Upload the DATA ZIP, not the earlier code ZIP.")

if DATA_SOURCE == "upload" and selected_zip.resolve() != cached_dataset.resolve():
    cached_dataset.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(selected_zip, cached_dataset)
    selected_zip = cached_dataset

DATA_ZIP = LOCAL_DATA_DIR / "Shida-LOSO-ANKLE.zip"
source_digest = file_sha256(selected_zip)
if not DATA_ZIP.exists() or file_sha256(DATA_ZIP) != source_digest:
    shutil.copy2(selected_zip, DATA_ZIP)
DATA_DIGEST = source_digest
print(f"Dataset ready: {len(raw_files)} Raw workbooks; {DATA_ZIP.stat().st_size / 1024**2:.1f} MiB")
print("SHA256:", DATA_DIGEST)


## 3 Load the verified model and reporting code

Run these embedded cells once per session. The controller below uses complete subject-wise validation and the same seven model families as the successful smoke test.


In [ ]:
#@title Data loader and participant split checks
write_module('data.py', r'''"""Read the paired per-participant Raw workbooks in Shida-LOSO-ANKLE."""
from __future__ import annotations

from dataclasses import dataclass
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import re
from zipfile import ZipFile

import numpy as np
import pandas as pd


TASKS = {
    "apleft": "AP_Left", "apright": "AP_Right",
    "longlef": "Vertical_Left", "longleft": "Vertical_Left",
    "longrig": "Vertical_Right", "longright": "Vertical_Right",
    "verticalleft": "Vertical_Left", "verticalright": "Vertical_Right",
    "mlleft": "ML_Left", "mlright": "ML_Right",
}
TASK_ORDER = ["AP_Left", "AP_Right", "Vertical_Left", "Vertical_Right", "ML_Left", "ML_Right"]
RAW_PATTERN = re.compile(r"(?:^|/)Raw/([^/]+)/([^/]+\.xlsx)$", re.I)
SUBJECT_PATTERN = re.compile(r"(?:SUB|UB)(\d+)", re.I)


@dataclass
class Dataset:
    task: str
    x: np.ndarray                       # [cycles, 100 time points, 1 feature]
    y: np.ndarray
    groups: np.ndarray                  # participant ID for EVERY cycle
    cycles: pd.DataFrame                # source and cycle-position provenance
    fingerprint: str

    @property
    def subjects(self):
        return sorted(np.unique(self.groups).tolist())

    def subset(self, subjects):
        mask = np.isin(self.groups, subjects)
        if not mask.any():
            raise ValueError("Empty participant subset")
        fingerprint = sha256((self.fingerprint + repr(sorted(subjects))).encode()).hexdigest()
        return Dataset(self.task, self.x[mask], self.y[mask], self.groups[mask],
                       self.cycles.loc[mask].reset_index(drop=True), fingerprint)


def canonical_subject(text):
    matches = SUBJECT_PATTERN.findall(str(text))
    ids = {f"SUB{int(value):02d}" for value in matches}
    if len(ids) != 1:
        raise ValueError(f"Expected one unambiguous participant in {text!r}; got {ids}")
    return ids.pop()


def workbook_arrays(payload, filename, subject, expected_timesteps=100):
    """Keep the original in/out column order. GRF headers lack trial identifiers."""
    with pd.ExcelFile(BytesIO(payload), engine="openpyxl") as book:
        if not {"in", "out"}.issubset(book.sheet_names):
            raise ValueError(f"{filename}: paired 'in' and 'out' sheets are required")
        inputs = pd.read_excel(book, sheet_name="in", header=0)
        targets = pd.read_excel(book, sheet_name="out", header=0)
    if inputs.shape != targets.shape or inputs.shape[0] != expected_timesteps or not inputs.shape[1]:
        raise ValueError(f"{filename}: expected matched ({expected_timesteps}, cycles) sheets; "
                         f"got in={inputs.shape}, out={targets.shape}")
    for header in inputs.columns:
        if canonical_subject(header) != subject:
            raise ValueError(f"{filename}: input column belongs to a different participant: {header}")
    try:
        x = inputs.to_numpy(dtype=np.float32).T[:, :, None]
        y = targets.to_numpy(dtype=np.float32).T[:, :, None]
    except (ValueError, TypeError) as exc:
        raise ValueError(f"{filename}: nonnumeric waveform value") from exc
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise ValueError(f"{filename}: missing/nonfinite values; no silent zero filling or exclusion is allowed")
    metadata = [{"subject": subject, "cycle_id": f"{subject}:C{i+1:03d}",
                 "column_position_1based": i + 1, "input_header": str(inputs.columns[i]),
                 "target_header": str(targets.columns[i]), "source_workbook": filename}
                for i in range(len(x))]
    return x, y, metadata


def load_datasets(source, requested_tasks=None):
    """Accept the uploaded ZIP, its extracted root, or the Raw directory itself.

    Never load the preassembled Train files: they repeat subjects across folds
    and some supplied input matrices were concatenated along the wrong axis.
    """
    source = Path(source).expanduser().resolve()
    requested = set(requested_tasks or TASK_ORDER)
    unknown = requested - set(TASK_ORDER)
    if unknown:
        raise ValueError(f"Unknown tasks: {sorted(unknown)}")
    archive = None
    if source.is_file() and source.suffix.lower() == ".zip":
        archive = ZipFile(source)
        names = sorted(n for n in archive.namelist() if RAW_PATTERN.search(n))
        reader = archive.read
    elif source.is_dir():
        paths = {p.as_posix(): p for p in source.rglob("*.xlsx") if RAW_PATTERN.search(p.as_posix())}
        names = sorted(paths)
        reader = lambda name: paths[name].read_bytes()
    else:
        raise ValueError(f"Expected an existing ZIP or directory: {source}")
    accum, seen, fingerprints = {}, set(), {}
    try:
        for name in names:
            match = RAW_PATTERN.search(name)
            task = TASKS.get(re.sub(r"[^a-z]", "", match.group(1).lower()))
            if task not in requested:
                continue
            subject = canonical_subject(match.group(2))
            if (task, subject) in seen:
                raise ValueError(f"Duplicate Raw workbook for {task}/{subject}; provide one dataset root")
            seen.add((task, subject))
            payload = reader(name)
            relative_name = "Raw/" + name.split("/")[-2] + "/" + name.split("/")[-1]
            x, y, metadata = workbook_arrays(payload, relative_name, subject)
            accum.setdefault(task, []).append((subject, x, y, metadata))
            fingerprints.setdefault(task, sha256()).update(relative_name.encode() + sha256(payload).digest())
    finally:
        if archive is not None:
            archive.close()
    absent = requested - set(accum)
    if absent:
        raise ValueError(f"No Raw workbooks found for {sorted(absent)} in {source}")
    result = {}
    for task in TASK_ORDER:
        if task not in accum:
            continue
        records = sorted(accum[task], key=lambda item: item[0])
        if len(records) < 3:
            raise ValueError(f"{task}: at least three participants are required for nested LOSO")
        result[task] = Dataset(
            task, np.concatenate([r[1] for r in records]), np.concatenate([r[2] for r in records]),
            np.concatenate([np.repeat(r[0], len(r[1])) for r in records]),
            pd.DataFrame([m for r in records for m in r[3]]), fingerprints[task].hexdigest())
    return result


def nested_indices(groups, outer_subject):
    """Yield train, validation, test indices; one entire subject in each holdout."""
    groups = np.asarray(groups)
    subjects = sorted(np.unique(groups).tolist())
    if outer_subject not in subjects or len(subjects) < 3:
        raise ValueError("Unknown outer participant or fewer than three participants")
    test = np.flatnonzero(groups == outer_subject)
    for validation_subject in subjects:
        if validation_subject == outer_subject:
            continue
        validation = np.flatnonzero(groups == validation_subject)
        train = np.flatnonzero((groups != outer_subject) & (groups != validation_subject))
        assert_subject_separation(groups, train, validation, test)
        yield validation_subject, train, validation, test


def assert_subject_separation(groups, train, validation, test):
    combined = np.concatenate([train, validation, test])
    if not np.array_equal(np.sort(combined), np.arange(len(groups))):
        raise ValueError("Each cycle must appear exactly once across the three partitions")
    sets = [set(np.asarray(groups)[idx]) for idx in [train, validation, test]]
    if any(not s for s in sets) or sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]:
        raise ValueError("Participant overlap or an empty train/validation/test partition")


def audit_datasets(datasets):
    tasks, participants, cycles, splits = [], [], [], []
    for task, ds in datasets.items():
        tasks.append({"task": task, "n_participants": len(ds.subjects), "n_cycles": len(ds.x),
                      "timepoints": ds.x.shape[1], "features": ds.x.shape[2],
                      "missing_input_values": int((~np.isfinite(ds.x)).sum()),
                      "missing_target_values": int((~np.isfinite(ds.y)).sum()),
                      "subjects": ds.subjects, "fingerprint": ds.fingerprint})
        cycles.extend(ds.cycles.assign(task=task).to_dict("records"))
        for subject in ds.subjects:
            participants.append({"task": task, "subject": subject,
                                 "n_cycles": int(np.sum(ds.groups == subject))})
            for val_subject, tr, va, te in nested_indices(ds.groups, subject):
                splits.append({"task": task, "outer_test_subject": subject,
                               "inner_validation_subject": val_subject,
                               "inner_training_subjects": sorted(set(ds.groups[tr])),
                               "n_train_cycles": len(tr), "n_validation_cycles": len(va),
                               "n_test_cycles": len(te)})
    return {"tasks": tasks, "participants": participants, "cycles": cycles, "splits": splits}
''')


In [ ]:
#@title Nested validation protocol and participant metrics
write_module('protocol.py', r'''"""Backend-independent nested LOSO protocol, metrics, and resumable records."""
from __future__ import annotations

from hashlib import sha256
import json
from pathlib import Path
import numpy as np
import pandas as pd

from data import Dataset, nested_indices


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def stable_seed(seed, *labels):
    text = "|".join(map(str, (seed, *labels)))
    return int.from_bytes(sha256(text.encode()).digest()[:4], "big") % (2**31 - 1)


def parameter_hash(parameters):
    return sha256(json.dumps(parameters, sort_keys=True).encode()).hexdigest()


def scale_from_training(train, *others, enabled=False):
    """Only training inputs determine scale. Targets retain their original units."""
    if enabled:
        mean = np.mean(train, axis=(0, 1), keepdims=True, dtype=np.float64)
        std = np.std(train, axis=(0, 1), keepdims=True, dtype=np.float64)
        std = np.where(std < 1e-8, 1.0, std)
    else:
        mean = np.zeros((1, 1, train.shape[-1]))
        std = np.ones_like(mean)
    arrays = [((arr - mean) / std).astype(np.float32) for arr in (train,) + others]
    return arrays, {"enabled": enabled, "mean": mean.reshape(-1).tolist(),
                    "std": std.reshape(-1).tolist()}


def cycle_metrics(truth, prediction):
    if truth.shape != prediction.shape or not np.isfinite(prediction).all():
        raise ValueError("Invalid prediction shape or nonfinite predictions")
    error = truth.astype(np.float64) - prediction.astype(np.float64)
    rmse = np.sqrt(np.mean(error**2, axis=(1, 2)))
    mae = np.mean(np.abs(error), axis=(1, 2))
    span = np.ptp(truth.astype(np.float64), axis=1).reshape(-1)
    nrmse = np.divide(rmse, span, out=np.full_like(rmse, np.nan), where=span > 1e-12)
    return pd.DataFrame({"rmse": rmse, "nrmse": nrmse, "mae": mae,
                         "target_range": span, "nrmse_defined": span > 1e-12})


def evaluate_candidate(ds, outer_subject, parameters, backend, seed, directory, scale_input=False):
    """Score one candidate equally across inner validation participants.

    The backend only receives inner train/validation data. Outer test arrays
    are never supplied to tuning, early stopping, or input scaling.
    """
    directory = Path(directory)
    signature = parameter_hash(parameters)
    records = []
    for validation_subject, tr, va, _ in nested_indices(ds.groups, outer_subject):
        path = directory / f"inner_{validation_subject}.json"
        if path.exists():
            record = read_json(path)
            if record["parameter_hash"] != signature:
                raise ValueError(f"Cached trial has different parameters: {path}")
            records.append(record)
            continue
        (xtr, xva), scaler = scale_from_training(ds.x[tr], ds.x[va], enabled=scale_input)
        fit_seed = stable_seed(seed, ds.task, outer_subject, validation_subject, "inner")
        fit = backend.fit_inner(xtr, ds.y[tr], xva, ds.y[va], parameters, fit_seed,
                                directory / f"history_{validation_subject}.csv")
        metric = cycle_metrics(ds.y[va], fit["prediction"])
        record = {"outer_subject": outer_subject, "validation_subject": validation_subject,
                  "training_subjects": sorted(set(ds.groups[tr])), "n_training_cycles": len(tr),
                  "n_validation_cycles": len(va), "seed": fit_seed,
                  "parameter_hash": signature, "scaler": scaler,
                  "best_epoch": int(fit["best_epoch"]), "epochs_run": int(fit["epochs_run"]),
                  "validation_mae": float(metric.mae.mean()),
                  "validation_rmse": float(metric.rmse.mean())}
        write_json(path, record)
        records.append(record)
    score = float(np.mean([r["validation_mae"] for r in records]))
    if not np.isfinite(score):
        raise ValueError("Nonfinite inner objective")
    summary = {"parameters": parameters, "inner_subject_mae": score, "folds": records}
    write_json(directory / "inner_summary.json", summary)
    return summary


def refit_and_evaluate(ds, outer_subject, parameters, inner_summary, backend, seed,
                       directory, architecture, scale_input=False):
    """Refit on ALL outer-training participants for an inner-derived epoch budget."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    tr = np.flatnonzero(ds.groups != outer_subject)
    te = np.flatnonzero(ds.groups == outer_subject)
    if len(te) == 0 or set(ds.groups[tr]) & set(ds.groups[te]):
        raise ValueError("Invalid outer subject separation")
    if set(r["validation_subject"] for r in inner_summary["folds"]) != set(ds.groups[tr]):
        raise ValueError("The winning trial did not validate every outer-training participant")
    if any(r["outer_subject"] != outer_subject for r in inner_summary["folds"]):
        raise ValueError("Inner summary belongs to a different outer fold")
    if parameter_hash(inner_summary["parameters"]) != parameter_hash(parameters):
        raise ValueError("Refit parameters differ from the selected inner trial")
    # Predefined rule: ceiling of median inner best epoch; outer test never selects epochs.
    epochs = max(1, int(np.ceil(np.median([r["best_epoch"] for r in inner_summary["folds"]]))))
    (xtr, xte), scaler = scale_from_training(ds.x[tr], ds.x[te], enabled=scale_input)
    refit_seed = stable_seed(seed, ds.task, outer_subject, "refit")
    model = backend.fit_final(xtr, ds.y[tr], parameters, refit_seed, epochs,
                              directory / "final_history.csv")
    # This is the first point where the fitted outer model sees test inputs.
    prediction = backend.predict(model, xte)
    metrics = cycle_metrics(ds.y[te], prediction)
    metadata = ds.cycles.iloc[te].reset_index(drop=True)
    per_cycle = pd.concat([metadata, metrics], axis=1)
    per_cycle.insert(0, "task", ds.task)
    per_cycle.insert(1, "architecture", architecture)
    per_cycle.insert(2, "seed", seed)
    per_cycle.to_csv(directory / "cycle_metrics.csv", index=False)
    np.savez_compressed(directory / "predictions.npz", measured=ds.y[te], predicted=prediction,
                        cycle_ids=metadata.cycle_id.to_numpy(dtype=str),
                        subject_ids=ds.groups[te], gait_phase=np.linspace(0, 100, ds.y.shape[1]))
    backend.save(model, directory / "model.keras")
    backend.clear()
    summary = {"task": ds.task, "architecture": architecture, "seed": seed,
               "subject": outer_subject, "n_cycles": len(te), "n_train_cycles": len(tr),
               "n_train_subjects": len(set(ds.groups[tr])), "epochs": epochs,
               "refit_seed": refit_seed, "rmse": float(metrics.rmse.mean()),
               "nrmse": float(metrics.nrmse.mean()) if metrics.nrmse.notna().any() else None,
               "mae": float(metrics.mae.mean()),
               "undefined_nrmse_cycles": int((~metrics.nrmse_defined).sum()),
               "inner_subject_mae": inner_summary["inner_subject_mae"],
               "parameters": parameters, "input_scaler": scaler,
               "epoch_rule": "ceiling of median best epoch across winning inner LOSO fits"}
    # Written last: a completed outer fold can be safely skipped on resume.
    write_json(directory / "result.json", summary)
    return summary


def summarize_subject_results(records, directory, bootstrap_seed=2026, resamples=2000):
    """Bootstrap participants; repeated seeds are averaged within participant first."""
    directory = Path(directory)
    df = pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, (list, dict))}
                       for r in records])
    if df.empty:
        return
    df.to_csv(directory / "outer_subject_metrics_by_seed.csv", index=False)
    averaged = df.groupby(["task", "architecture", "subject"], as_index=False).agg(
        rmse=("rmse", "mean"), nrmse=("nrmse", "mean"), mae=("mae", "mean"),
        n_seeds=("seed", "nunique"), rmse_seed_sd=("rmse", "std"),
        nrmse_seed_sd=("nrmse", "std"), mae_seed_sd=("mae", "std"))
    averaged.to_csv(directory / "outer_subject_metrics_seed_average.csv", index=False)
    rows = []
    for (task, arch), group in averaged.groupby(["task", "architecture"]):
        for metric in ["rmse", "nrmse", "mae"]:
            values = group[metric].dropna().to_numpy()
            if not len(values):
                continue
            rng = np.random.default_rng(stable_seed(bootstrap_seed, task, arch, metric))
            if len(values) >= 2:
                boot = np.median(rng.choice(values, size=(resamples, len(values)), replace=True), axis=1)
                low, high = np.quantile(boot, [0.025, 0.975])
            else:
                low = high = np.nan
            rows.append({"task": task, "architecture": arch, "metric": metric,
                         "n_participants": len(values), "min_seeds_per_subject": int(group.n_seeds.min()),
                         "max_seeds_per_subject": int(group.n_seeds.max()),
                         "median": float(np.median(values)), "q25": float(np.quantile(values, .25)),
                         "q75": float(np.quantile(values, .75)),
                         "median_ci95_lower": low, "median_ci95_upper": high,
                         "ci_method": "participant percentile bootstrap after averaging seeds"})
    pd.DataFrame(rows).to_csv(directory / "cohort_summary.csv", index=False)
''')


In [ ]:
#@title LSTM GRU Conv1D LSTM and Bayesian tuning
write_module('models.py', r'''"""TensorFlow model definitions and Bayesian selection over complete inner LOSO.

TensorFlow/KerasTuner are imported only during training. Audit and data tests
therefore work in a lightweight environment without a GPU or TensorFlow.
"""
from __future__ import annotations

import gc
from pathlib import Path
import numpy as np

from protocol import evaluate_candidate, read_json, stable_seed, write_json


def training_dependencies():
    try:
        import tensorflow as tf
        import keras_tuner as kt
    except ImportError as exc:
        raise RuntimeError("Training requires TensorFlow and keras-tuner. Install requirements.txt "
                           "in a Python 3.11 environment; audit mode needs only requirements-audit.txt.") from exc
    return tf, kt


def configure_tensorflow(seed, threads=0):
    tf, _ = training_dependencies()
    if threads:
        tf.config.threading.set_inter_op_parallelism_threads(threads)
        tf.config.threading.set_intra_op_parallelism_threads(threads)
    for device in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(device, True)
    tf.keras.utils.set_random_seed(seed)
    tf.config.experimental.enable_op_determinism()
    return {"tensorflow": tf.__version__, "devices": [d.name for d in tf.config.list_physical_devices()]}


def search_space(architecture, profile):
    _, kt = training_dependencies()
    hp = kt.HyperParameters()
    if profile == "smoke":
        hp.Fixed("num_layers", 1)
        hp.Fixed("activation", "tanh")
        hp.Fixed("optimizer", "adam")
        hp.Fixed("learning_rate", 0.001)
        hp.Fixed("loss", "mae")
        hp.Fixed("encoder_units", 4)
        hp.Fixed("decoder_units", 4)
        hp.Fixed("units_0", 4)
        return hp
    max_layers = 10 if profile == "original" else 3
    hp.Int("num_layers", 1, max_layers)
    # The original GRU code registers tanh first under the shared HP name.
    hp.Choice("activation", ["relu", "tanh"] if architecture == "conv1d_lstm" and profile == "original" else ["tanh"])
    hp.Choice("optimizer", ["adam", "sgd", "rmsprop"] if profile == "original" else ["adam"])
    hp.Choice("learning_rate", [0.0001, 0.001, 0.01] if profile == "original" else [0.0001, 0.001])
    hp.Choice("loss", ["mse", "mae", "huber"] if architecture == "gru" else ["mse", "mae"])
    hp.Fixed("encoder_units", (32 if architecture == "gru" else 100) if profile == "original" else 32)
    if architecture != "gru":
        hp.Choice("decoder_units", list(range(32, 513, 32)) if profile == "original" else [32, 64, 128])
    unit_options = (list(range(32, 513, 32)) if architecture == "gru" else list(range(100, 513, 32)))
    if profile != "original":
        unit_options = [32, 64, 128]
    for layer in range(max_layers):
        with hp.conditional_scope("num_layers", list(range(layer + 1, max_layers + 1))):
            hp.Choice(f"units_{layer}", unit_options)
    return hp


def build_model(architecture, timesteps, parameters, profile="original"):
    tf, _ = training_dependencies()
    layers = tf.keras.layers
    model = tf.keras.Sequential(name=architecture)
    model.add(layers.Input(shape=(timesteps, 1)))
    activation = parameters["activation"]
    if architecture == "conv1d_lstm":
        model.add(layers.Conv1D(4 if profile == "smoke" else 64, 3, activation="relu"))
        model.add(layers.MaxPooling1D(pool_size=2))
    if architecture in {"lstm", "conv1d_lstm"}:
        model.add(layers.LSTM(parameters["encoder_units"], activation=activation))
        model.add(layers.RepeatVector(timesteps))
        model.add(layers.LSTM(parameters["decoder_units"], activation=activation, return_sequences=True))
        for layer in range(parameters["num_layers"]):
            model.add(layers.LSTM(parameters[f"units_{layer}"], activation=activation, return_sequences=True))
        model.add(layers.TimeDistributed(layers.Dense(1)))
    elif architecture == "gru":
        model.add(layers.GRU(parameters["encoder_units"], activation=activation, return_sequences=True))
        for layer in range(parameters["num_layers"]):
            model.add(layers.GRU(parameters[f"units_{layer}"], activation=activation, return_sequences=True))
        model.add(layers.Dense(1))
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    optimizer = {"adam": tf.keras.optimizers.Adam, "sgd": tf.keras.optimizers.SGD,
                 "rmsprop": tf.keras.optimizers.RMSprop}[parameters["optimizer"]](
                     learning_rate=parameters["learning_rate"])
    loss = tf.keras.losses.Huber() if parameters["loss"] == "huber" else parameters["loss"]
    model.compile(optimizer=optimizer, loss=loss, metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")])
    return model


class TensorFlowBackend:
    def __init__(self, architecture, timesteps, profile, epochs, patience, batch_size, verbose):
        self.tf, _ = training_dependencies()
        self.architecture, self.timesteps, self.profile = architecture, timesteps, profile
        self.epochs, self.patience = epochs, patience
        self.batch_size, self.verbose = batch_size, verbose

    def clear(self):
        self.tf.keras.backend.clear_session()
        gc.collect()

    def fresh(self, parameters, seed):
        self.clear()
        self.tf.keras.utils.set_random_seed(seed)
        return build_model(self.architecture, self.timesteps, parameters, self.profile)

    def fit_inner(self, x_train, y_train, x_validation, y_validation, parameters, seed, history_path):
        history_path = Path(history_path)
        history_path.parent.mkdir(parents=True, exist_ok=True)
        model = self.fresh(parameters, seed)
        tf = self.tf
        stop = tf.keras.callbacks.EarlyStopping(monitor="val_mae", mode="min",
                                               patience=self.patience, min_delta=0,
                                               restore_best_weights=True)
        try:
            history = model.fit(x_train, y_train,
                                validation_data=(x_validation, y_validation),
                                epochs=self.epochs, batch_size=self.batch_size, shuffle=True,
                                callbacks=[stop, tf.keras.callbacks.TerminateOnNaN(),
                                           tf.keras.callbacks.CSVLogger(str(history_path))],
                                verbose=self.verbose)
            values = np.asarray(history.history["val_mae"], dtype=float)
            if not len(values) or not np.isfinite(values).all():
                raise ValueError("Nonfinite inner training/validation history; trial must fail")
            best_epoch = int(np.argmin(values)) + 1
            prediction = self.predict(model, x_validation)
            return {"prediction": prediction, "best_epoch": best_epoch, "epochs_run": len(values)}
        finally:
            del model
            self.clear()

    def fit_final(self, x_train, y_train, parameters, seed, epochs, history_path):
        """No validation split or validation-driven callback in the outer refit."""
        model = self.fresh(parameters, seed)
        tf = self.tf
        history = model.fit(x_train, y_train, epochs=epochs, batch_size=self.batch_size,
                            shuffle=True, callbacks=[tf.keras.callbacks.TerminateOnNaN(),
                                                      tf.keras.callbacks.CSVLogger(str(history_path))],
                            verbose=self.verbose)
        if len(history.history["loss"]) != epochs or not np.isfinite(history.history["loss"]).all():
            raise ValueError("Outer refit did not complete with finite losses")
        return model

    def predict(self, model, inputs):
        return np.asarray(model.predict(inputs, batch_size=self.batch_size, verbose=0))

    def save(self, model, path):
        model.save(path)
        lines = []
        model.summary(print_fn=lambda line, **kwargs: lines.append(line))
        Path(path).with_suffix(".summary.txt").write_text("\n".join(lines), encoding="utf-8")


def select_hyperparameters(ds, outer_subject, architecture, backend, seed, directory,
                           profile, max_trials, scale_input, resume=False):
    """A Bayesian trial is a COMPLETE inner LOSO, never just one validation subject."""
    _, kt = training_dependencies()
    directory = Path(directory)
    trial_root = directory / "candidate_fits"
    trial_root.mkdir(parents=True, exist_ok=True)

    class InnerLOSOTuner(kt.Tuner):
        def run_trial(self, trial, *args, **kwargs):
            parameters = dict(trial.hyperparameters.values)
            trial_dir = trial_root / trial.trial_id
            print(f"  {ds.task}/{architecture}/{outer_subject}: trial {trial.trial_id}", flush=True)
            try:
                summary = evaluate_candidate(ds, outer_subject, parameters, backend, seed,
                                             trial_dir, scale_input)
            except (ValueError, backend.tf.errors.ResourceExhaustedError) as exc:
                write_json(trial_dir / "failure.json", {"error": str(exc), "parameters": parameters})
                # Do not average only the successful validation participants.
                raise kt.errors.FailedTrialError(str(exc)) from exc
            return {"inner_subject_mae": summary["inner_subject_mae"]}

    hp = search_space(architecture, profile)
    oracle = kt.oracles.BayesianOptimizationOracle(
        objective=kt.Objective("inner_subject_mae", direction="min"),
        max_trials=max_trials, num_initial_points=min(3, max_trials),
        hyperparameters=hp, allow_new_entries=False, tune_new_entries=False,
        seed=stable_seed(seed, ds.task, architecture, outer_subject, "oracle"),
        max_retries_per_trial=0, max_consecutive_failed_trials=3)
    tuner = InnerLOSOTuner(oracle=oracle, hypermodel=None, directory=str(directory),
                          project_name="tuner", overwrite=not resume)
    tuner.search()
    best = tuner.oracle.get_best_trials(num_trials=1)
    if not best or best[0].score is None or not np.isfinite(best[0].score):
        raise RuntimeError("No successful full inner LOSO trial; inspect failure.json files")
    trial = best[0]
    summary = read_json(trial_root / trial.trial_id / "inner_summary.json")
    write_json(directory / "selection.json", {
        "trial_id": trial.trial_id, "inner_subject_mae": summary["inner_subject_mae"],
        "parameters": summary["parameters"], "search_profile": profile,
        "n_inner_folds": len(summary["folds"])})
    return summary["parameters"], summary
''')


In [ ]:
#@title Training runner and resume checks
write_module('nested_loso.py', r'''#!/usr/bin/env python3
"""Nested subject-wise LOSO for the uploaded Shida ankle/GRF archive.

Start with --mode audit (the default). See README.md before a full training run.
"""
from __future__ import annotations

import argparse
from datetime import datetime, timezone
from hashlib import sha256
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import sys

import pandas as pd

from data import TASK_ORDER, audit_datasets, load_datasets
from protocol import read_json, refit_and_evaluate, summarize_subject_results, write_json


def software_versions():
    versions = {"python": platform.python_version(), "platform": platform.platform()}
    for package in ["numpy", "pandas", "openpyxl", "tensorflow", "keras", "keras-tuner", "scikit-learn"]:
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    return versions


def code_fingerprint():
    digest = sha256()
    for name in ["data.py", "models.py", "protocol.py", "nested_loso.py"]:
        digest.update((Path(__file__).parent / name).read_bytes())
    return digest.hexdigest()


def ensure_configuration(directory, config, resume):
    directory = Path(directory)
    path = directory / "experiment_config.json"
    if path.exists():
        if not resume:
            raise ValueError(f"Existing experiment: {directory}. Use --resume or a new --output directory.")
        if read_json(path) != config:
            raise ValueError(f"Data/code/settings/software changed in {directory}; use a new output directory")
    else:
        if directory.exists() and any(directory.iterdir()):
            raise ValueError(f"Nonempty experiment without a configuration: {directory}")
        write_json(path, config)


def write_audit(audit, directory, args):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / "dataset_audit.json", {
        "tasks": audit["tasks"], "source": str(Path(args.data).resolve()),
        "paired_by": "same column position in a participant workbook's in/out sheets",
        "preassembled_train_files_used": False,
        "zero_filling_used": False, "smoke_test": args.smoke_test})
    pd.DataFrame(audit["participants"]).to_csv(directory / "participants.csv", index=False)
    pd.DataFrame(audit["cycles"]).to_csv(directory / "cycles.csv", index=False)
    write_json(directory / "nested_splits.json", audit["splits"])


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", required=True, help="Shida-LOSO-ANKLE.zip or extracted dataset/Raw directory")
    parser.add_argument("--output", default="nested_loso_results")
    parser.add_argument("--mode", choices=["audit", "train"], default="audit")
    parser.add_argument("--tasks", nargs="+", choices=TASK_ORDER, default=TASK_ORDER)
    parser.add_argument("--architectures", nargs="+", choices=["lstm", "gru", "conv1d_lstm"],
                        default=["lstm", "gru", "conv1d_lstm"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[42])
    parser.add_argument("--max-trials", type=int, default=10)
    parser.add_argument("--epochs", type=int, default=1000)
    parser.add_argument("--patience", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--search-profile", choices=["original", "compact"], default="original")
    parser.add_argument("--scale-input", action="store_true", help="Optional training-only standardization; off by default")
    parser.add_argument("--outer-subjects", nargs="+", help="Run selected outer folds while retaining ALL inner subjects")
    parser.add_argument("--resume", action="store_true", help="Resume only if data, code, settings and versions match")
    parser.add_argument("--smoke-test", action="store_true", help="One outer fold, three subjects, tiny networks, one epoch; NOT research results")
    parser.add_argument("--verbose", type=int, choices=[0, 1, 2], default=0)
    parser.add_argument("--threads", type=int, default=0, help="TensorFlow CPU threads; 0 keeps its defaults")
    args = parser.parse_args(argv)
    if min(args.max_trials, args.epochs, args.batch_size) < 1 or args.patience < 0 or args.threads < 0:
        parser.error("Trials, epochs and batch size must be positive; patience/threads must be nonnegative")
    if any(s < 0 or s >= 2**31 for s in args.seeds):
        parser.error("Seeds must be in [0, 2**31)")
    for attr in ["tasks", "architectures", "seeds"]:
        if len(getattr(args, attr)) != len(set(getattr(args, attr))):
            parser.error(f"Duplicate entries in --{attr}")
    if args.outer_subjects:
        args.outer_subjects = [s.upper() for s in args.outer_subjects]
    if args.smoke_test:
        args.mode, args.tasks, args.seeds = "train", ["AP_Left"], [args.seeds[0]]
        args.search_profile, args.max_trials, args.epochs, args.patience = "smoke", 1, 1, 0
        args.outer_subjects = None
    return args


def main(argv=None):
    args = parse_args(argv)
    # Configure environment BEFORE TensorFlow is imported.
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
    os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
    datasets = load_datasets(args.data, args.tasks)
    if args.smoke_test:
        datasets = {task: ds.subset(ds.subjects[:3]) for task, ds in datasets.items()}
    audit = audit_datasets(datasets)
    outer_selection = {}
    for task, ds in datasets.items():
        selected = ds.subjects[:1] if args.smoke_test else (args.outer_subjects or ds.subjects)
        missing = set(selected) - set(ds.subjects)
        if missing:
            raise ValueError(f"{task}: requested outer participants absent: {sorted(missing)}")
        outer_selection[task] = selected
        print(f"{task}: {len(ds.subjects)} participants, {len(ds.x)} cycles, "
              f"{ds.x.shape[1]} time points; {len(selected)} outer folds", flush=True)
    fit_count = sum(len(outer_selection[t]) * (args.max_trials * (len(ds.subjects) - 1) + 1)
                    for t, ds in datasets.items()) * len(args.architectures) * len(args.seeds)
    print(f"Planned upper bound: {fit_count:,} model fits (inner trials plus outer refits).", flush=True)
    output = Path(args.output).expanduser().resolve()
    write_audit(audit, output / "audit", args)
    if args.mode == "audit":
        print(f"Audit passed. No training performed. Reports: {output / 'audit'}")
        return 0

    from models import TensorFlowBackend, configure_tensorflow, select_hyperparameters
    hardware = configure_tensorflow(args.seeds[0], args.threads)
    versions = software_versions()
    settings = {key: getattr(args, key) for key in ["max_trials", "epochs", "patience", "batch_size",
                                                   "search_profile", "scale_input", "smoke_test", "threads"]}
    run_info = {"utc_started": datetime.now(timezone.utc).isoformat(), "arguments": vars(args),
                "software": versions, "hardware": hardware, "code_fingerprint": code_fingerprint(),
                "expected_fit_upper_bound": fit_count}
    write_json(output / "run_metadata.json", run_info)
    experiment_dirs = []
    # Validate every requested configuration before spending time training.
    for task, ds in datasets.items():
        for architecture in args.architectures:
            for seed in args.seeds:
                directory = output / task / architecture / f"seed_{seed}"
                config = {"task": task, "data_fingerprint": ds.fingerprint,
                          "architecture": architecture, "seed": seed, "settings": settings,
                          "software": {k: v for k, v in versions.items() if k != "platform"},
                          "code_fingerprint": run_info["code_fingerprint"]}
                ensure_configuration(directory, config, args.resume)
                experiment_dirs.append(directory)
    for task, ds in datasets.items():
        for architecture in args.architectures:
            backend = TensorFlowBackend(architecture, ds.x.shape[1], args.search_profile,
                                        args.epochs, args.patience, args.batch_size, args.verbose)
            for seed in args.seeds:
                experiment = output / task / architecture / f"seed_{seed}"
                for outer_subject in outer_selection[task]:
                    directory = experiment / f"outer_{outer_subject}"
                    if (directory / "result.json").exists():
                        print(f"Completed fold retained: {task}/{architecture}/{seed}/{outer_subject}", flush=True)
                        continue
                    print(f"Training {task}/{architecture}/seed={seed}/outer={outer_subject}", flush=True)
                    params, inner_summary = select_hyperparameters(
                        ds, outer_subject, architecture, backend, seed, directory,
                        args.search_profile, args.max_trials, args.scale_input, args.resume)
                    refit_and_evaluate(ds, outer_subject, params, inner_summary, backend, seed,
                                       directory, architecture, args.scale_input)
    # Scope summary to the task/architecture/seed combinations requested in this invocation.
    records = [read_json(p) for directory in experiment_dirs
               for p in sorted(directory.glob("outer_*/result.json"))]
    summarize_subject_results(records, output)
    coverage = [{"task": task, "architecture": arch, "seed": seed,
                 "available_subjects": ds.subjects,
                 "completed_subjects": sorted(r["subject"] for r in records
                                              if r["task"] == task and r["architecture"] == arch and r["seed"] == seed)}
                for task, ds in datasets.items() for arch in args.architectures for seed in args.seeds]
    write_json(output / "coverage.json", {"smoke_test": args.smoke_test, "experiments": coverage})
    print(f"Finished. Participant-level metrics and coverage report: {output}", flush=True)
    if args.smoke_test:
        print("SMOKE TEST ONLY: these results are not suitable for the manuscript.", flush=True)
    return 0


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except (ValueError, RuntimeError) as exc:
        print(f"ERROR: {exc}", file=sys.stderr)
        raise SystemExit(2)
''')


In [ ]:
#@title Protocol regression tests
write_module('tests/test_protocol.py', r'''"""Leakage and aggregation tests. No TensorFlow dependency and no model claims."""
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

import tempfile
import unittest
import numpy as np
import pandas as pd

from data import Dataset, assert_subject_separation, canonical_subject, nested_indices
from protocol import (cycle_metrics, evaluate_candidate, read_json, refit_and_evaluate,
                      scale_from_training, summarize_subject_results)


def fixture():
    # Unequal cycle counts detect accidental cycle weighting of the inner score.
    groups = np.array(["SUB01"] * 2 + ["SUB02"] * 3 + ["SUB03"] * 1 + ["SUB04"] * 4)
    participant_value = np.array([int(s[-2:]) for s in groups], dtype=np.float32)
    x = np.repeat(participant_value[:, None, None], 8, axis=1)
    y = x + np.linspace(0, 1, 8, dtype=np.float32)[None, :, None]
    metadata = pd.DataFrame({"subject": groups, "cycle_id": [f"c{i}" for i in range(len(groups))]})
    return Dataset("AP_Left", x, y, groups, metadata, "synthetic-test-only")


class RecordingBackend:
    """A deterministic test double; it does not train or approximate a neural net."""
    def __init__(self):
        self.inner_calls = []
        self.final_calls = []

    def fit_inner(self, xtr, ytr, xva, yva, parameters, seed, path):
        self.inner_calls.append((xtr.copy(), ytr.copy(), xva.copy(), yva.copy()))
        subject = int(xva[0, 0, 0])
        return {"prediction": yva + subject, "best_epoch": subject, "epochs_run": subject + 1}

    def fit_final(self, xtr, ytr, parameters, seed, epochs, path):
        self.final_calls.append((xtr.copy(), ytr.copy(), epochs))
        return np.mean(ytr, axis=0, keepdims=True)

    def predict(self, model, inputs):
        return np.repeat(model, len(inputs), axis=0)

    def save(self, model, path):
        Path(path).write_text("TEST DOUBLE ONLY", encoding="utf-8")

    def clear(self):
        pass


class ProtocolTests(unittest.TestCase):
    def test_all_nested_folds_keep_whole_subjects(self):
        ds = fixture()
        for outer in ds.subjects:
            inner_subjects = []
            for val, tr, va, te in nested_indices(ds.groups, outer):
                inner_subjects.append(val)
                self.assertEqual(set(ds.groups[te]), {outer})
                self.assertEqual(set(ds.groups[va]), {val})
                self.assertEqual(set(ds.groups[tr]), set(ds.subjects) - {outer, val})
                self.assertEqual(sorted(np.concatenate([tr, va, te]).tolist()), list(range(len(ds.x))))
            self.assertEqual(set(inner_subjects), set(ds.subjects) - {outer})

    def test_overlap_is_rejected(self):
        ds = fixture()
        with self.assertRaises(ValueError):
            assert_subject_separation(ds.groups, np.array([0, 2, 5, 6, 7, 8, 9]),
                                      np.array([1]), np.array([3, 4]))

    def test_subject_tokens_do_not_confuse_sub01_with_sub10(self):
        self.assertEqual(canonical_subject("SUB01_off_UB01_walk"), "SUB01")
        self.assertEqual(canonical_subject("SUB10_walk"), "SUB10")
        with self.assertRaises(ValueError):
            canonical_subject("SUB01_UB10_walk")

    def test_scaler_never_uses_validation_or_test_values(self):
        tr = np.array([0., 2.], dtype=np.float32).reshape(1, 2, 1)
        va = np.full((1, 2, 1), 1000., dtype=np.float32)
        (scaled_tr, scaled_va), record = scale_from_training(tr, va, enabled=True)
        self.assertEqual(record["mean"], [1.])
        self.assertEqual(record["std"], [1.])
        np.testing.assert_allclose(scaled_tr.reshape(-1), [-1., 1.])
        np.testing.assert_allclose(scaled_va, 999.)

    def test_candidate_equal_subject_weight_and_outer_exclusion(self):
        ds, backend = fixture(), RecordingBackend()
        with tempfile.TemporaryDirectory() as tmp:
            summary = evaluate_candidate(ds, "SUB01", {}, backend, 42, tmp)
        self.assertEqual(len(backend.inner_calls), 3)
        self.assertAlmostEqual(summary["inner_subject_mae"], 3.)  # mean(2,3,4), not cycle-weighted
        for xtr, ytr, xva, yva in backend.inner_calls:
            self.assertNotIn(1., xtr)
            self.assertNotIn(1., xva)
            self.assertFalse(set(xtr[:, 0, 0]) & set(xva[:, 0, 0]))

    def test_poisoning_outer_data_cannot_change_inner_selection(self):
        original, changed = fixture(), fixture()
        changed.x[changed.groups == "SUB01"] = 1e7
        changed.y[changed.groups == "SUB01"] = -1e7
        with tempfile.TemporaryDirectory() as a, tempfile.TemporaryDirectory() as b:
            one = evaluate_candidate(original, "SUB01", {}, RecordingBackend(), 42, a)
            two = evaluate_candidate(changed, "SUB01", {}, RecordingBackend(), 42, b)
        self.assertEqual(one, two)

    def test_completed_inner_fits_resume_without_retraining(self):
        ds, backend = fixture(), RecordingBackend()
        with tempfile.TemporaryDirectory() as tmp:
            first = evaluate_candidate(ds, "SUB01", {"a": 1}, backend, 42, tmp)
            again = evaluate_candidate(ds, "SUB01", {"a": 1}, backend, 42, tmp)
            self.assertEqual(first, again)
            self.assertEqual(len(backend.inner_calls), 3)
            with self.assertRaises(ValueError):
                evaluate_candidate(ds, "SUB01", {"a": 2}, backend, 42, tmp)

    def test_outer_refit_uses_all_development_subjects_and_inner_epochs(self):
        ds, backend = fixture(), RecordingBackend()
        with tempfile.TemporaryDirectory() as tmp:
            selected = evaluate_candidate(ds, "SUB01", {}, backend, 42, Path(tmp) / "inner")
            result = refit_and_evaluate(ds, "SUB01", {}, selected, backend, 42, Path(tmp) / "outer", "test")
            xtr, ytr, epochs = backend.final_calls[0]
            self.assertEqual(set(xtr[:, 0, 0]), {2., 3., 4.})
            self.assertEqual(len(xtr), 8)
            self.assertEqual(epochs, 3)
            self.assertEqual(result["n_cycles"], 2)
            saved = np.load(Path(tmp) / "outer" / "predictions.npz")
            self.assertEqual(set(saved["subject_ids"]), {"SUB01"})
            self.assertEqual(saved["measured"].shape, (2, 8, 1))
            self.assertEqual(read_json(Path(tmp) / "outer" / "result.json")["epochs"], 3)

    def test_undefined_nrmse_is_not_replaced_by_arbitrary_denominator(self):
        truth = np.array([[[2.], [2.]], [[0.], [2.]]])
        result = cycle_metrics(truth, truth + 1)
        self.assertTrue(np.isnan(result.nrmse.iloc[0]))
        self.assertEqual(result.nrmse.iloc[1], .5)
        self.assertEqual(result.rmse.iloc[0], 1.)

    def test_repeated_seeds_do_not_inflate_participant_count(self):
        records = [{"task": "AP_Left", "architecture": "lstm", "subject": subject,
                    "seed": seed, "rmse": value, "nrmse": value / 10, "mae": value}
                   for subject, value in [("SUB01", 1.), ("SUB02", 3.)] for seed in [1, 2, 3]]
        with tempfile.TemporaryDirectory() as tmp:
            summarize_subject_results(records, tmp, resamples=30)
            summary = pd.read_csv(Path(tmp) / "cohort_summary.csv")
            self.assertTrue((summary.n_participants == 2).all())
            self.assertTrue((summary.min_seeds_per_subject == 3).all())
            self.assertEqual(summary.loc[summary.metric == "rmse", "median"].iloc[0], 2.)


if __name__ == "__main__":
    unittest.main()
''')


In [ ]:
#@title Cohort data contract and coordinate transforms
write_module('revision_data.py', r'''"""Strict cohort contract and coordinate controls; no guessed anatomical data."""
from dataclasses import dataclass, replace
from hashlib import sha256
from pathlib import Path
import json
import numpy as np
import pandas as pd
from data import Dataset, load_datasets
from protocol import write_json, read_json

MODES = ('absolute', 'translation', 'pelvis', 'aligned', 'body', 'combined')
AXIS = {'AP': 0, 'ML': 1, 'Vertical': 2}
REQUIRED_CONTEXT = ['cohort_id', 'dataset_role', 'ankle_units', 'grf_units',
                    'coordinate_definition', 'motion_capture_system', 'force_platforms',
                    'kinematics_sampling_hz', 'force_sampling_hz', 'walking_protocol',
                    'harmonization_description', 'cycle_definition']

def digest_file(path):
    h = sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def bool_column(series):
    mapping = {'true': True, '1': True, 'yes': True, 'false': False, '0': False, 'no': False}
    v = series.astype(str).str.lower().str.strip().map(mapping)
    if v.isna().any():
        raise ValueError('include must contain explicit true/false values')
    return v.astype(bool)

def read_table(path, columns):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if set(columns) - set(df):
        raise ValueError(f'{path}: missing columns {set(columns) - set(df)}')
    for col in set(columns) - {'exclusion_reason'}:
        if df[col].str.strip().eq('').any():
            raise ValueError(f'{path}: blank required {col}')
    return df

@dataclass
class Cohort:
    ds: Dataset
    context: dict
    participants: pd.DataFrame
    trials: pd.DataFrame
    ledger: pd.DataFrame
    arrays: dict

def legacy_cohort(archive, task):
    ds = load_datasets(archive, [task])[task]
    meta = ds.cycles.copy()
    meta['participant_id'] = meta.subject
    meta['trial_id'] = 'unknown'  # Never invent trial IDs from cycle positions.
    meta['visit'] = 'internal'
    meta['side'] = task.split('_')[-1].lower()
    meta['include'] = True
    meta['exclusion_reason'] = ''
    ds = replace(ds, cycles=meta)
    context = {k: 'unknown' for k in REQUIRED_CONTEXT}
    context.update(cohort_id='Shida_internal', dataset_role='internal', ankle_units='mm',
                   grf_units='N/kg', units_verified=False,
                   coordinate_definition='Original single component; physical frame unverified',
                   source='Paired Raw sheets; manuscript units assumed, not independently verified')
    people = pd.DataFrame({'participant_id': ds.subjects})
    return Cohort(ds, context, people, pd.DataFrame(), meta, {})

def load_cohort(directory, task, visits=None, role=None):
    """Read one side/component cohort bundle. Excluded cycles need no waveform."""
    root = Path(directory)
    context = read_json(root / 'cohort.json')
    if any(k not in context or context[k] is None for k in REQUIRED_CONTEXT):
        raise ValueError('Complete cohort.json; use "unknown" explicitly for unavailable descriptive fields')
    if role and context['dataset_role'] != role:
        raise ValueError('Wrong dataset_role')
    if context['ankle_units'] not in ('m', 'mm') or context['grf_units'] != 'N/kg':
        raise ValueError('Convert ankle units to m/mm and GRF to N/kg before making the bundle')
    if context.get('task') != task:
        raise ValueError('cohort.json task differs from requested component/side')
    people = read_table(root / 'participants.csv', ['participant_id'])
    trials = read_table(root / 'trials.csv', ['participant_id', 'trial_id', 'visit', 'side', 'include', 'exclusion_reason'])
    ledger = read_table(root / 'cycles.csv', ['cycle_id', 'participant_id', 'trial_id', 'visit', 'side', 'include', 'exclusion_reason'])
    if people.participant_id.duplicated().any() or ledger.cycle_id.duplicated().any():
        raise ValueError('Participant and cycle IDs must be unique within this cohort bundle')
    if not people.participant_id.str.fullmatch(r'[A-Za-z0-9_-]+').all():
        raise ValueError('Use letters, digits, underscores, or hyphens in participant IDs')
    keys = ['participant_id', 'trial_id', 'visit', 'side']
    if trials.duplicated(keys).any():
        raise ValueError('Duplicate trial record')
    for frame in (trials, ledger):
        frame['include'] = bool_column(frame.include)
        if frame.loc[~frame.include, 'exclusion_reason'].str.strip().eq('').any():
            raise ValueError('Every excluded trial/cycle requires a reason')
        if set(frame.participant_id) - set(people.participant_id):
            raise ValueError('Participant missing from participants.csv')
        if not set(frame.visit).issubset({'baseline', 'post_intervention', 'internal'}):
            raise ValueError('Use baseline, post_intervention, or internal visit labels')
        if set(frame.side) != {task.split('_')[-1].lower()}:
            raise ValueError('Each bundle must describe just its selected side')
    joined = ledger.merge(trials[keys + ['include']], on=keys, how='left', suffixes=('', '_trial'), validate='many_to_one')
    if joined.include_trial.isna().any() or (joined.include & ~joined.include_trial.astype(bool)).any():
        raise ValueError('Every cycle must link to a trial; an excluded trial cannot have an included cycle')
    with np.load(root / 'waveforms.npz', allow_pickle=False) as f:
        arrays = {k: f[k] for k in f.files}
    ids = arrays.pop('cycle_ids').astype(str)
    valid = ledger[ledger.include].copy().set_index('cycle_id', drop=False)
    if len(set(ids)) != len(ids) or set(ids) != set(valid.index):
        raise ValueError('NPZ cycle_ids must match ALL included cycles, one-to-one, before visit filtering')
    meta = valid.loc[ids].reset_index(drop=True)
    n = len(ids)
    for k, arr in arrays.items():
        if len(arr) != n or not np.issubdtype(arr.dtype, np.number) or not np.isfinite(arr).all():
            raise ValueError(f'{k}: wrong cycle count or nonfinite/nonnumeric values')
    if 'ankle_xyz' in arrays:
        xyz = arrays['ankle_xyz']
        if xyz.shape != (n, 100, 3) or arrays.get('grf_xyz', np.empty(0)).shape != xyz.shape:
            raise ValueError('ankle_xyz and grf_xyz must both be [N,100,3]')
        if context.get('canonical_axis_order') != ['forward', 'left', 'up']:
            raise ValueError('Map both kinematics and GRFs into the same right-handed [forward,left,up] laboratory axes')
        if arrays.get('direction_xyz', np.empty(0)).shape != (n, 3):
            raise ValueError('Provide a force-independent horizontal walking direction for every cycle')
        basis = gait_basis(arrays['direction_xyz'])
        axis = AXIS[task.split('_')[0]]
        x = xyz[:, :, axis:axis + 1]
        # SAME gait-frame targets for ALL coordinate variants, enabling fair comparisons.
        y = np.einsum('nti,nij->ntj', arrays['grf_xyz'], basis)[:, :, axis:axis + 1]
    else:
        x, y = arrays.get('x'), arrays.get('y')
        if x is None or y is None or x.shape != (n, 100, 1) or y.shape != x.shape:
            raise ValueError('Scalar bundles require x and y, each [N,100,1]')
    take = np.ones(n, bool) if visits is None else meta.visit.isin(visits).to_numpy()
    if not take.any():
        raise ValueError('No valid cycles for the prespecified visit policy')
    meta = meta.loc[take].reset_index(drop=True)
    meta['subject'] = meta.participant_id
    fingerprint = sha256(''.join(digest_file(root / f) for f in
                         ['cohort.json', 'participants.csv', 'trials.csv', 'cycles.csv', 'waveforms.npz']).encode()).hexdigest()
    fingerprint = sha256((fingerprint + repr(visits)).encode()).hexdigest()
    ds = Dataset(task, x[take].astype('float32'), y[take].astype('float32'),
                 meta.participant_id.to_numpy(str), meta, fingerprint)
    return Cohort(ds, context, people, trials, ledger, {k: v[take] for k, v in arrays.items()})

def gait_basis(direction):
    forward = np.array(direction, float, copy=True)
    if forward.ndim != 2 or forward.shape[1] != 3 or not np.isfinite(forward).all():
        raise ValueError('Invalid direction vectors')
    if np.any(np.abs(forward[:, 2]) > 1e-6):
        raise ValueError('Walking directions must be projected onto the horizontal plane')
    length = np.linalg.norm(forward, axis=1)
    if np.any(length < 1e-8):
        raise ValueError('Zero walking direction; supply verified direction from pelvis progression/protocol')
    forward /= length[:, None]
    up = np.tile([0., 0., 1.], (len(forward), 1))
    left = np.cross(up, forward)
    return np.stack([forward, left, up], axis=2)

def coordinate_dataset(cohort, mode='absolute', translation_m=None, rotation_deg=0):
    if mode not in MODES:
        raise ValueError(f'Unknown coordinate mode {mode}')
    ds, a = cohort.ds, cohort.arrays
    units = cohort.context['ankle_units']
    factor = 1000. if units == 'mm' else 1.
    axis = AXIS[ds.task.split('_')[0]]
    if 'ankle_xyz' not in a:
        if mode not in ('absolute', 'translation') or rotation_deg:
            raise ValueError('This analysis requires verified 3D/pelvis/body-scale data; scalar inputs cannot supply it')
        x = ds.x.astype(float).copy()
        if translation_m is not None:
            if not cohort.context.get('units_verified', False):
                raise ValueError('Verify physical units before metre-based perturbations')
            x += np.asarray(translation_m)[axis] * factor
        if mode == 'translation':
            x -= x[:, :1, :]
    else:
        x = a['ankle_xyz'].astype(float).copy()
        pelvis = a.get('pelvis_xyz')
        pelvis = None if pelvis is None else pelvis.astype(float).copy()
        direction = a['direction_xyz'].astype(float).copy()
        if rotation_deg:
            theta = np.deg2rad(rotation_deg)
            R = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]])
            x = x @ R.T
            direction = direction @ R.T
            if pelvis is not None:
                pelvis = pelvis @ R.T
        if translation_m is not None:
            if not cohort.context.get('units_verified', False):
                raise ValueError('Verify physical units before metre-based perturbations')
            offset = np.asarray(translation_m, float) * factor
            if offset.shape != (3,):
                raise ValueError('Translation requires three metre offsets')
            x += offset
            if pelvis is not None:
                pelvis += offset
        if mode in ('pelvis', 'combined'):
            if pelvis is None or pelvis.shape != x.shape:
                raise ValueError('Synchronized pelvis_xyz [N,100,3] is required')
            x -= pelvis
        elif mode in ('translation', 'aligned', 'body'):
            x -= x[:, :1, :]
        if mode in ('aligned', 'combined'):
            x = np.einsum('nti,nij->ntj', x, gait_basis(direction))
        if mode in ('body', 'combined'):
            if not cohort.context.get('units_verified', False) or not cohort.context.get('body_scale_definition'):
                raise ValueError('Verify coordinate units and prespecify height or leg-length definition')
            height = a.get('body_scale_m')
            if height is None or height.shape != (len(x),) or np.any(height <= 0):
                raise ValueError('Positive body_scale_m [N], with a fixed definition, is required')
            x /= height[:, None, None] * factor
        x = x[:, :, axis:axis + 1]
    fingerprint = sha256((ds.fingerprint + repr((mode, translation_m, rotation_deg))).encode()).hexdigest()
    return replace(ds, x=x.astype('float32'), fingerprint=fingerprint)

def write_templates(directory):
    """Blank data contract, not fabricated external observations."""
    root = Path(directory)
    root.mkdir(parents=True, exist_ok=True)
    for name, columns in {
        'participants.csv': ['participant_id', 'global_subject_id', 'age', 'sex', 'medication_state', 'HY_stage', 'intervention_group'],
        'trials.csv': ['participant_id', 'trial_id', 'visit', 'side', 'include', 'exclusion_reason'],
        'cycles.csv': ['cycle_id', 'participant_id', 'trial_id', 'visit', 'side', 'include', 'exclusion_reason'],
    }.items():
        if not (root / name).exists():
            pd.DataFrame(columns=columns).to_csv(root / name, index=False)
    context = {k: None for k in REQUIRED_CONTEXT}
    context.update(task=None, units_verified=False, harmonization_verified=False, participant_disjointness_verified=False,
                   canonical_axis_order=['forward', 'left', 'up'], body_scale_definition=None)
    if not (root / 'cohort.json').exists():
        write_json(root / 'cohort.json', context)
    return root

def harmonize_input_units(cohort, training_units):
    """Deterministic physical-unit conversion, never fitted on external data."""
    units = cohort.context['ankle_units']
    if units not in ('m', 'mm') or training_units not in ('m', 'mm'):
        raise ValueError('Known internal and external length units are required')
    if units == training_units:
        return cohort
    multiplier = 1000. if units == 'm' else .001
    arrays = dict(cohort.arrays)
    for key in ['ankle_xyz', 'pelvis_xyz']:
        if key in arrays:
            arrays[key] = arrays[key] * multiplier
    ds = replace(cohort.ds, x=cohort.ds.x * multiplier)
    return replace(cohort, ds=ds, arrays=arrays,
                   context={**cohort.context, 'ankle_units': training_units})

def pack_scalar_workbooks(input_xlsx, target_xlsx, cycle_map_csv, participants_csv,
                          trials_csv, cohort_json, output):
    """Convert explicitly mapped old-format Excel cycles; never infer participants."""
    import shutil
    root = Path(output)
    root.mkdir(parents=True, exist_ok=True)
    ledger = read_table(cycle_map_csv, ['cycle_id', 'participant_id', 'trial_id', 'visit', 'side',
                                       'include', 'exclusion_reason'])
    if {'input_column', 'target_column'} - set(ledger):
        raise ValueError('Explicit input_column and target_column mappings are required')
    accepted = ledger[bool_column(ledger.include)]
    inputs = pd.read_excel(input_xlsx, engine='openpyxl')
    targets = pd.read_excel(target_xlsx, engine='openpyxl')
    if inputs.shape[0] != 100 or targets.shape[0] != 100:
        raise ValueError('Prepare matched complete cycles at 100 normalized gait phases before packing; no padding/resampling is guessed')
    xs, ys = [], []
    for row in accepted.itertuples():
        x = inputs[row.input_column].to_numpy(dtype='float32')
        y = targets[row.target_column].to_numpy(dtype='float32')
        if not np.isfinite(x).all() or not np.isfinite(y).all():
            raise ValueError(f'{row.cycle_id}: missing values; document exclusion or fix the source, never zero-fill')
        xs.append(x[:, None]); ys.append(y[:, None])
    if not xs:
        raise ValueError('No included cycles')
    np.savez_compressed(root / 'waveforms.npz', cycle_ids=accepted.cycle_id.to_numpy(str), x=np.stack(xs), y=np.stack(ys))
    ledger.to_csv(root / 'cycles.csv', index=False)
    for source, filename in [(participants_csv, 'participants.csv'), (trials_csv, 'trials.csv'), (cohort_json, 'cohort.json')]:
        if Path(source).resolve() != (root / filename).resolve():
            shutil.copy2(source, root / filename)
    context = read_json(root / 'cohort.json')
    load_cohort(root, context['task'], role=context['dataset_role'])
    return root
''')


In [ ]:
#@title Baseline models and joint subject-wise selection
write_module('revision_models.py', r'''"""Subject-wise model-family selection and four explicit comparator families."""
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning
from protocol import cycle_metrics, stable_seed, scale_from_training, write_json, read_json, parameter_hash

BASELINES = ('mean_waveform', 'ridge', 'phase_only', 'shallow_mlp')
DEEP = ('lstm', 'gru', 'conv1d_lstm')

def participant_mean(y, groups):
    return np.mean([y[groups == g].mean(axis=0) for g in np.unique(groups)], axis=0)

def fourier_basis(timesteps, harmonics):
    # Endpoints include 0 and 100% consistently with the supplied waveforms.
    phase = np.linspace(0, 1, timesteps)
    return np.column_stack([np.ones(timesteps)] +
        [f(2 * np.pi * k * phase) for k in range(1, harmonics + 1) for f in (np.sin, np.cos)])

def baseline_grid(family, smoke=False):
    if family == 'mean_waveform':
        return [{}]
    if family == 'ridge':
        return [{'alpha': v} for v in ([1.] if smoke else [0.01, 0.1, 1., 10., 100.])]
    if family == 'phase_only':
        return [{'harmonics': v} for v in ([3] if smoke else [1, 3, 5, 10])]
    if family == 'shallow_mlp':
        return [{'units': u, 'alpha': a, 'max_iter': 30 if smoke else 1000}
                for u in ([4] if smoke else [16, 32, 64])
                for a in ([0.01] if smoke else [0.0001, 0.01])]
    raise ValueError(family)

def fit_baseline(family, x, y, groups, parameters, seed):
    obj = {'family': family, 'shape': list(y.shape[1:]), 'parameters': parameters}
    if family == 'mean_waveform':
        obj['waveform'] = participant_mean(y, groups)
    elif family == 'phase_only':
        basis = fourier_basis(y.shape[1], parameters['harmonics'])
        # Equal participant weighting. Uses phase alone, never ankle positions.
        coef = np.linalg.lstsq(basis, participant_mean(y, groups), rcond=None)[0]
        obj['waveform'] = basis @ coef
    else:
        flat = x.reshape(len(x), -1)
        scaler = StandardScaler().fit(flat)  # Training rows only.
        z = scaler.transform(flat)
        target = y.reshape(len(y), -1)
        if family == 'ridge':
            model = Ridge(alpha=parameters['alpha'])
            weights = np.array([1. / np.sum(groups == g) for g in groups])
            weights *= len(weights) / weights.sum()
            model.fit(z, target, sample_weight=weights)
            obj['fit_weighting'] = 'equal participants'
        elif family == 'shallow_mlp':
            # One hidden layer; training-loss optimizer convergence, NO random validation split.
            model = MLPRegressor(hidden_layer_sizes=(parameters['units'],), activation='relu',
                                 solver='lbfgs', alpha=parameters['alpha'], max_iter=parameters['max_iter'],
                                 max_fun=50000, early_stopping=False, random_state=seed)
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always', ConvergenceWarning)
                model.fit(z, target)
            obj['convergence_warnings'] = [str(w.message) for w in caught]
            obj['fit_weighting'] = 'equal cycles; validation score remains equal participants'
        else:
            raise ValueError(family)
        obj.update(model=model, scaler=scaler)
    return obj

def predict_baseline(obj, x):
    if 'waveform' in obj:
        return np.repeat(obj['waveform'][None], len(x), axis=0)
    z = obj['scaler'].transform(x.reshape(len(x), -1))
    return obj['model'].predict(z).reshape((len(x), *obj['shape']))

def backend(family, ds, cfg):
    from models import TensorFlowBackend
    return TensorFlowBackend(family, ds.x.shape[1], cfg['search_profile'], cfg['epochs'],
                             cfg['patience'], cfg['batch_size'], cfg['verbose'])

def cv_candidate(ds, family, parameters, cfg, directory):
    """Only the development Dataset is passed; outer/external arrays are absent."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    signature = parameter_hash(parameters)
    rows = []
    for val in ds.subjects:
        path = directory / f'validation_{val}.json'
        if path.exists():
            row = read_json(path)
            if row['parameter_hash'] != signature or row['data_fingerprint'] != ds.fingerprint:
                raise ValueError('Candidate cache changed')
            rows.append(row)
            continue
        tr, va = ds.groups != val, ds.groups == val
        if not tr.any():
            raise ValueError('At least two development participants are required')
        seed = stable_seed(cfg['seed'], ds.task, family, val, 'inner')
        print(f'    {family}: validation {val}', flush=True)
        if family in DEEP:
            b = backend(family, ds, cfg)
            (xt, xv), _ = scale_from_training(ds.x[tr], ds.x[va], enabled=cfg['scale_input'])
            fit = b.fit_inner(xt, ds.y[tr], xv, ds.y[va], parameters, seed, directory / f'history_{val}.csv')
            prediction, epoch = fit['prediction'], fit['best_epoch']
            convergence = []
        else:
            model = fit_baseline(family, ds.x[tr], ds.y[tr], ds.groups[tr], parameters, seed)
            prediction, epoch = predict_baseline(model, ds.x[va]), None
            convergence = model.get('convergence_warnings', [])
        metric = cycle_metrics(ds.y[va], np.asarray(prediction))
        row = {'validation_subject': val, 'training_subjects': sorted(set(ds.groups[tr])),
               'n_validation_cycles': int(va.sum()), 'parameter_hash': signature,
               'data_fingerprint': ds.fingerprint, 'mae': float(metric.mae.mean()),
               'rmse': float(metric.rmse.mean()), 'best_epoch': epoch, 'convergence_warnings': convergence}
        write_json(path, row)
        rows.append(row)
    score = float(np.mean([r['mae'] for r in rows]))
    if not np.isfinite(score):
        raise ValueError('Nonfinite full subject-wise validation objective')
    result = {'family': family, 'parameters': parameters, 'inner_subject_mae': score, 'folds': rows,
              'epochs': max(1, int(np.ceil(np.median([r['best_epoch'] for r in rows])))) if family in DEEP else None}
    write_json(directory / 'summary.json', result)
    return result

def tune_family(ds, family, cfg, directory):
    directory = Path(directory)
    done = directory / 'selected.json'
    if done.exists():
        return read_json(done)
    directory.mkdir(parents=True, exist_ok=True)
    if family not in DEEP:
        candidates = [cv_candidate(ds, family, p, cfg, directory / f'candidate_{i:03d}')
                      for i, p in enumerate(baseline_grid(family, cfg.get('smoke', False)))]
        selected = min(candidates, key=lambda r: (r['inner_subject_mae'], parameter_hash(r['parameters'])))
    else:
        from models import training_dependencies, search_space
        tf, kt = training_dependencies()
        class SubjectTuner(kt.Tuner):
            def run_trial(self, trial, *args, **kwargs):
                try:
                    result = cv_candidate(ds, family, dict(trial.hyperparameters.values), cfg,
                                          directory / 'candidates' / trial.trial_id)
                except (ValueError, tf.errors.ResourceExhaustedError) as exc:
                    write_json(directory / 'candidates' / trial.trial_id / 'failure.json', {'error': str(exc)})
                    raise kt.errors.FailedTrialError(str(exc)) from exc
                return {'inner_subject_mae': result['inner_subject_mae']}
        hp = search_space(family, cfg['search_profile'])
        oracle = kt.oracles.BayesianOptimizationOracle(objective=kt.Objective('inner_subject_mae', 'min'),
            max_trials=cfg['max_trials'], num_initial_points=min(3, cfg['max_trials']),
            hyperparameters=hp, allow_new_entries=False, tune_new_entries=False,
            seed=stable_seed(cfg['seed'], ds.task, family, 'oracle'), max_consecutive_failed_trials=3)
        tuner = SubjectTuner(oracle=oracle, hypermodel=None, directory=str(directory), project_name='tuner', overwrite=False)
        tuner.search()
        best = tuner.oracle.get_best_trials(1)
        if not best or best[0].score is None or not np.isfinite(best[0].score):
            raise ValueError(f'No complete successful candidate for {family}')
        selected = read_json(directory / 'candidates' / best[0].trial_id / 'summary.json')
    write_json(done, selected)
    return selected

def fit_frozen_model(ds, selected, cfg, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    descriptor = directory / 'model.json'
    if descriptor.exists():
        return read_json(descriptor)
    family = selected['family']
    seed = stable_seed(cfg['seed'], ds.task, family, 'final_refit')
    if family in DEEP:
        b = backend(family, ds, cfg)
        (x,), scaler = scale_from_training(ds.x, enabled=cfg['scale_input'])
        model = b.fit_final(x, ds.y, selected['parameters'], seed, selected['epochs'], directory / 'history.csv')
        b.save(model, directory / 'model.keras')
        b.clear()
        result = {'family': family, 'file': 'model.keras', 'input_scaler': scaler}
    else:
        model = fit_baseline(family, ds.x, ds.y, ds.groups, selected['parameters'], seed)
        joblib.dump(model, directory / 'model.joblib')
        result = {'family': family, 'file': 'model.joblib',
                  'convergence_warnings': model.get('convergence_warnings', [])}
    result.update(parameters=selected['parameters'], epochs=selected['epochs'], seed=seed,
                  n_training_participants=len(ds.subjects), n_training_cycles=len(ds.x),
                  training_participants=ds.subjects, data_fingerprint=ds.fingerprint)
    write_json(descriptor, result)
    return result

def predict_saved(directory, x):
    directory = Path(directory)
    info = read_json(directory / 'model.json')
    if info['family'] in DEEP:
        from models import training_dependencies
        tf, _ = training_dependencies()
        scaler = info['input_scaler']
        z = (x - np.array(scaler['mean'])[None, None]) / np.array(scaler['std'])[None, None]
        model = tf.keras.models.load_model(directory / info['file'], compile=False)
        try:
            return np.asarray(model.predict(z, verbose=0))
        finally:
            del model
            tf.keras.backend.clear_session()
    # Only load models created by this notebook, whose file hashes are frozen.
    return np.asarray(predict_baseline(joblib.load(directory / info['file']), x))

def mean_ensemble(predictions):
    """Average predictions BEFORE errors: models are not independent participants."""
    values = [np.asarray(p) for p in predictions]
    if not values or any(v.shape != values[0].shape or not np.isfinite(v).all() for v in values):
        raise ValueError('Ensemble members require identical finite cycle/time/component arrays')
    return np.mean(np.stack(values), axis=0)
''')


In [ ]:
#@title Participant confidence intervals and domain-shift reports
write_module('revision_reports.py', r'''"""Participant-level errors, trial-cluster intervals, and domain-shift accounting."""
from pathlib import Path
import numpy as np
import pandas as pd
from protocol import cycle_metrics, stable_seed, write_json

METRICS = ('rmse', 'nrmse', 'mae')

def prediction_table(ds, prediction, model):
    metrics = cycle_metrics(ds.y, prediction)
    table = pd.concat([ds.cycles.reset_index(drop=True), metrics], axis=1)
    table['model'] = model
    table['task'] = ds.task
    if 'participant_id' not in table:
        table['participant_id'] = ds.groups
    if 'visit' not in table:
        table['visit'] = 'internal'
    if 'trial_id' not in table:
        table['trial_id'] = 'unknown'
    # Pearson r is descriptive only; it does not replace error/baseline comparisons.
    centered_y = ds.y.reshape(len(ds.y), -1).astype(float)
    centered_p = prediction.reshape(len(prediction), -1).astype(float)
    centered_y -= centered_y.mean(axis=1, keepdims=True)
    centered_p -= centered_p.mean(axis=1, keepdims=True)
    denom = np.linalg.norm(centered_y, axis=1) * np.linalg.norm(centered_p, axis=1)
    table['waveform_r'] = np.divide((centered_y * centered_p).sum(axis=1), denom,
                                   out=np.full(len(denom), np.nan), where=denom > 1e-12)
    return table

def bootstrap_interval(values, seed, repeats, statistic=np.median):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    estimates = statistic(rng.choice(values, size=(repeats, len(values)), replace=True), axis=1)
    return tuple(np.quantile(estimates, [0.025, 0.975]))

def participant_table(cycles, repeats=2000, seed=2026):
    keys = ['task', 'model', 'participant_id', 'visit']
    if cycles.duplicated(keys + ['cycle_id']).any():
        raise ValueError('Duplicate predictions for the same participant/cycle/model: do not stack model replicates')
    rows = []
    for key, frame in cycles.groupby(keys, sort=True):
        known_trials = ~frame.trial_id.isin(['', 'unknown'])
        trial_ids = frame.trial_id.unique() if known_trials.all() else []
        row = dict(zip(keys, key))
        row.update(n_cycles=len(frame), n_trials=len(trial_ids) if len(trial_ids) else None,
                   undefined_nrmse_cycles=int(frame.nrmse.isna().sum()),
                   ci_method='trial-cluster percentile bootstrap; conditional on fitted model',
                   ci_status='estimated' if len(trial_ids) >= 2 else 'unavailable: fewer than two known trials')
        row['waveform_r'] = frame.waveform_r.mean() if 'waveform_r' in frame else np.nan
        for metric in METRICS:
            row[metric] = frame[metric].mean()
            row[f'{metric}_ci95_low'] = row[f'{metric}_ci95_high'] = np.nan
            if len(trial_ids) < 2:
                continue
            # Cluster resampling preserves all cycles within each sampled trial.
            groups = [frame.loc[frame.trial_id == t, metric].dropna().to_numpy() for t in trial_ids]
            rng = np.random.default_rng(stable_seed(seed, *key, metric))
            estimates = []
            for _ in range(repeats):
                draw = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
                if len(draw):
                    estimates.append(draw.mean())
            if estimates:
                row[f'{metric}_ci95_low'], row[f'{metric}_ci95_high'] = np.quantile(estimates, [.025, .975])
        rows.append(row)
    return pd.DataFrame(rows)

def cohort_table(participants, repeats=2000, seed=2026):
    rows = []
    for key, frame in participants.groupby(['task', 'model', 'visit'], sort=True):
        for metric in METRICS:
            v = frame[metric].dropna().to_numpy()
            if not len(v):
                continue
            lo, hi = bootstrap_interval(v, stable_seed(seed, *key, metric), repeats)
            rows.append(dict(task=key[0], model=key[1], visit=key[2], metric=metric,
                n_participants=len(v), mean=float(v.mean()), median=float(np.median(v)),
                q25=float(np.quantile(v, .25)), q75=float(np.quantile(v, .75)),
                median_ci95_low=lo, median_ci95_high=hi,
                ci_method='participant percentile bootstrap; conditional on fitted pipeline'))
    return pd.DataFrame(rows)

def paired_comparisons(participants, reference='selected_deep', repeats=2000, seed=2026):
    rows = []
    base = participants[participants.model == reference]
    keys = ['task', 'visit', 'participant_id']
    for other in sorted(set(participants.model) - {reference}):
        joined = base.merge(participants[participants.model == other], on=keys, suffixes=('_deep', '_baseline'), validate='one_to_one')
        for (task, visit), frame in joined.groupby(['task', 'visit']):
            for metric in METRICS:
                # Positive = lower error in the deep model. Correlation is not the endpoint.
                delta = (frame[f'{metric}_baseline'] - frame[f'{metric}_deep']).dropna().to_numpy()
                lo, hi = bootstrap_interval(delta, stable_seed(seed, task, visit, other, metric), repeats)
                rows.append(dict(task=task, visit=visit, deep_model=reference, comparator=other, metric=metric,
                    n_paired_participants=len(delta), mean_improvement=float(delta.mean()) if len(delta) else np.nan,
                    median_improvement=float(np.median(delta)) if len(delta) else np.nan,
                    median_improvement_ci95_low=lo, median_improvement_ci95_high=hi,
                    interpretation='positive favors deep; descriptive intervals, no multiplicity-adjusted superiority claim'))
    return pd.DataFrame(rows)

def stratified_table(participants, people, columns, minimum=5, repeats=2000, seed=2026):
    if people.participant_id.duplicated().any():
        raise ValueError('One metadata row per participant is required')
    data = participants.merge(people, on='participant_id', how='left', validate='many_to_one')
    rows = []
    for col in columns:
        if col not in data:
            rows.append(dict(characteristic=col, stratum='unknown', status='metadata unavailable'))
            continue
        data[col] = data[col].replace('', 'unknown').fillna('unknown')
        for key, group in data.groupby(['task', 'model', 'visit', col], dropna=False):
            n = group.participant_id.nunique()
            status = 'reported' if n >= minimum and str(key[3]).lower() not in ('unknown', 'nan') else 'suppressed: sparse or unknown'
            for metric in METRICS:
                v = group[metric].dropna().to_numpy()
                sufficient = status == 'reported' and len(v) >= minimum
                lo, hi = bootstrap_interval(v, stable_seed(seed, *key, col, metric), repeats) if sufficient else (np.nan, np.nan)
                rows.append(dict(task=key[0], model=key[1], visit=key[2], characteristic=col, stratum=key[3],
                    n_participants=n, n_metric_defined=len(v), metric=metric, status=status if sufficient else 'suppressed: sparse or unknown',
                    median=float(np.median(v)) if sufficient else np.nan,
                    median_ci95_low=lo, median_ci95_high=hi))
    return pd.DataFrame(rows)

def save_reports(cycles, directory, cfg, people=None):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    cycles.to_csv(directory / 'cycle_metrics.csv', index=False)
    participants = participant_table(cycles, cfg['bootstrap_resamples'], cfg['bootstrap_seed'])
    participants.to_csv(directory / 'participant_metrics.csv', index=False)
    cohort_table(participants, cfg['bootstrap_resamples'], cfg['bootstrap_seed']).to_csv(directory / 'cohort_summary.csv', index=False)
    paired_comparisons(participants, repeats=cfg['bootstrap_resamples'], seed=cfg['bootstrap_seed']).to_csv(directory / 'paired_baseline_comparisons.csv', index=False)
    if people is not None:
        metadata = people.copy()
        if 'age' in metadata:
            ages = pd.to_numeric(metadata.age, errors='coerce')
            # Boundaries are in the locked prespecification, not chosen from errors.
            metadata['age_band'] = pd.cut(ages, bins=cfg['age_bin_edges'], right=False).astype(str)
        stratified_table(participants, metadata, cfg['strata'], cfg['minimum_stratum_n'],
                         cfg['bootstrap_resamples'], cfg['bootstrap_seed']).to_csv(directory / 'stratified_results.csv', index=False)
    return participants

def audit_external(cohort, visits, directory, internal_context):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    ledger = cohort.ledger.copy()
    ledger['analysis_included'] = ledger.include & ledger.visit.isin(visits)
    ledger['analysis_exclusion_reason'] = np.where(~ledger.include, ledger.exclusion_reason,
                                                  np.where(~ledger.visit.isin(visits), 'visit excluded by prespecified analysis policy', ''))
    ledger.to_csv(directory / 'cycle_flow.csv', index=False)
    trials = cohort.trials.copy()
    keys = ['participant_id', 'trial_id', 'visit', 'side']
    counts = ledger[ledger.analysis_included].groupby(keys).size().rename('valid_cycles').reset_index()
    trials = trials.merge(counts, on=keys, how='left')
    trials['valid_cycles'] = trials.valid_cycles.fillna(0).astype(int)
    trials['analysis_included'] = trials.include & trials.visit.isin(visits) & trials.valid_cycles.gt(0)
    trials['analysis_exclusion_reason'] = np.where(~trials.include, trials.exclusion_reason,
        np.where(~trials.visit.isin(visits), 'visit excluded by prespecified analysis policy',
        np.where(trials.valid_cycles.eq(0), 'no valid analyzed cycles', '')))
    trials.to_csv(directory / 'trial_flow.csv', index=False)
    roster = cohort.participants.copy()
    pc = ledger[ledger.analysis_included].groupby('participant_id').size()
    roster['valid_cycles'] = roster.participant_id.map(pc).fillna(0).astype(int)
    roster['analysis_exclusion_reason'] = np.where(roster.valid_cycles.eq(0),
        'no analyzed eligible cycles; see trial/cycle exclusion ledgers', '')
    roster.to_csv(directory / 'participant_flow.csv', index=False)
    for name, frame in [('cycle', ledger), ('trial', trials)]:
        frame.loc[~frame.analysis_included].groupby(['visit', 'analysis_exclusion_reason']).size().rename('count').reset_index().to_csv(directory / f'{name}_exclusion_reasons.csv', index=False)
    rows = []
    for visit in sorted(set(trials.visit) | set(ledger.visit)):
        l, t = ledger[ledger.visit == visit], trials[trials.visit == visit]
        rows.append(dict(visit=visit, n_participants_with_trials=t.participant_id.nunique(),
            n_trials_recorded=len(t), n_candidate_cycles_recorded=len(l),
            n_participants_analyzed=l.loc[l.analysis_included, 'participant_id'].nunique(),
            n_trials_analyzed=int(t.analysis_included.sum()), n_valid_cycles=int(l.analysis_included.sum())))
    pd.DataFrame(rows).to_csv(directory / 'cohort_flow_by_visit.csv', index=False)
    write_json(directory / 'cohort_flow_totals.json', {'n_participants_in_roster': len(roster),
        'n_participants_analyzed': int(roster.valid_cycles.gt(0).sum()),
        'n_participants_without_analyzed_cycles': int(roster.valid_cycles.eq(0).sum()),
        'n_trials_recorded': len(trials), 'n_trials_analyzed': int(trials.analysis_included.sum()),
        'n_valid_cycles': int(ledger.analysis_included.sum()), 'visit_policy': visits})
    keys = sorted(set(internal_context) | set(cohort.context))
    pd.DataFrame([{'characteristic': k, 'internal': str(internal_context.get(k, 'unknown')),
                   'external': str(cohort.context.get(k, 'unknown'))} for k in keys]).to_csv(directory / 'domain_shift_context.csv', index=False)
''')


In [ ]:
#@title Nested study final refit and locked external evaluation
write_module('revision_study.py', r'''"""Extended nested study -> full-cohort refit -> locked external domain-shift test."""
import argparse
from datetime import datetime, timezone
from dataclasses import replace
from pathlib import Path
import json
import importlib.metadata
import platform
import numpy as np
import pandas as pd
from protocol import write_json, read_json, parameter_hash, stable_seed
from revision_data import (legacy_cohort, load_cohort, coordinate_dataset, digest_file, MODES, harmonize_input_units)
from revision_models import BASELINES, DEEP, tune_family, fit_frozen_model, predict_saved, mean_ensemble
from revision_reports import save_reports, prediction_table, audit_external, bootstrap_interval

def default_config():
    return dict(seed=42, search_profile='original', max_trials=10, epochs=1000,
        patience=5, batch_size=32, verbose=0, scale_input=False, smoke=False,
        families=list(BASELINES + DEEP), coordinate_mode='absolute',
        architecture_rule='minimum complete inner-LOSO participant MAE; ties lexical family name',
        final_epoch_rule='ceiling of median best epoch from selected full-internal LOSO candidate',
        external_visits=['baseline'], external_status='previously_evaluated',
        include_loso_ensemble=False, ensemble_rule='equal-weight arithmetic mean of all outer-selected deep waveforms',
        bootstrap_resamples=2000, bootstrap_seed=2026, minimum_stratum_n=5,
        strata=['sex', 'medication_state', 'HY_stage', 'age_band', 'intervention_group'],
        age_bin_edges=[0, 65, 75, 120],
        translation_offsets_m=[[0.25, 0., 0.], [-0.25, 0., 0.], [0., 0.25, 0.], [0., -0.25, 0.]],
        rotation_angles_deg=[-15., 15.], run_perturbations=False,
        scope='one side/component per experiment; external visit summaries remain separate')

def validate_config(cfg):
    if set(default_config()) - set(cfg):
        raise ValueError('Incomplete configuration')
    if cfg['coordinate_mode'] not in MODES or cfg['external_status'] not in ('untouched_new_holdout', 'previously_evaluated'):
        raise ValueError('Invalid coordinates/external status')
    if not cfg['external_visits'] or set(cfg['external_visits']) - {'baseline', 'post_intervention'}:
        raise ValueError('Prespecify baseline and/or post_intervention; reports never pool visits')
    if not cfg['families'] or set(cfg['families']) - set(BASELINES + DEEP) or len(set(cfg['families'])) != len(cfg['families']):
        raise ValueError('Unknown or duplicate model family')
    if min(cfg['max_trials'], cfg['epochs'], cfg['batch_size']) < 1 or cfg['patience'] < 0:
        raise ValueError('Invalid training settings')
    if cfg['bootstrap_resamples'] < 100 or cfg['minimum_stratum_n'] < 2:
        raise ValueError('Use >=100 bootstrap draws and >=2 participants per reported stratum')
    if cfg['seed'] < 0 or cfg['seed'] >= 2**31:
        raise ValueError('Invalid seed')
    if np.any(np.diff(cfg['age_bin_edges']) <= 0):
        raise ValueError('Age bin edges must be strictly increasing')

def software():
    info = {'python': platform.python_version()}
    for name in ['numpy', 'pandas', 'scikit-learn', 'tensorflow', 'keras', 'keras-tuner', 'joblib']:
        try:
            info[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            info[name] = None
    return info

def source_hashes():
    root = Path(__file__).parent
    return {p.name: digest_file(p) for p in [root / name for name in
        ['data.py', 'protocol.py', 'models.py', 'revision_data.py', 'revision_models.py', 'revision_reports.py', 'revision_study.py']]}

def bind_experiment(root, ds, cfg):
    validate_config(cfg)
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    signature = {'config': cfg, 'task': ds.task, 'data_fingerprint': ds.fingerprint,
                 'subjects': ds.subjects, 'sources': source_hashes(), 'software': software()}
    path = root / 'prespecification.json'
    if path.exists():
        if read_json(path) != signature:
            raise ValueError('Prespecification, data, code, or software changed. Use the original settings or a new experiment.')
    else:
        if any(root.iterdir()):
            raise ValueError('Nonempty directory without a prespecification')
        write_json(path, signature)
    return signature

def choose_deep(selections):
    choices = [s for s in selections if s['family'] in DEEP]
    if not choices:
        raise ValueError('No deep family was evaluated')
    return min(choices, key=lambda s: (s['inner_subject_mae'], s['family']))

def run_nested(cohort, cfg, root, outer_subjects=None):
    root = Path(root)
    ds = coordinate_dataset(cohort, cfg['coordinate_mode'])
    bind_experiment(root, ds, cfg)
    selected = outer_subjects or ds.subjects
    if set(selected) - set(ds.subjects) or len(ds.subjects) < 3:
        raise ValueError('Unknown outer participant or fewer than three participants')
    for outer in selected:
        print(f'OUTER TEST = {outer}', flush=True)
        tr_ds = ds.subset([s for s in ds.subjects if s != outer])
        te_ds = ds.subset([outer])
        outer_dir = root / 'nested' / outer
        selections = []
        for family in cfg['families']:
            directory = outer_dir / family
            chosen = tune_family(tr_ds, family, cfg, directory / 'selection')
            selections.append(chosen)
            marker = directory / 'complete.json'
            if not marker.exists():
                model_dir = directory / 'refit'
                fit_frozen_model(tr_ds, chosen, cfg, model_dir)
                prediction = predict_saved(model_dir, te_ds.x)
                prediction_table(te_ds, prediction, family).to_csv(directory / 'cycles.csv', index=False)
                np.savez_compressed(directory / 'predictions.npz', cycle_ids=te_ds.cycles.cycle_id.to_numpy(str),
                                    measured=te_ds.y, predicted=prediction)
                write_json(marker, {'outer_subject': outer, 'family': family, 'inner_subject_mae': chosen['inner_subject_mae']})
        if set(cfg['families']) & set(DEEP):
            winner = choose_deep(selections)
            source = pd.read_csv(outer_dir / winner['family'] / 'cycles.csv', dtype={'participant_id': str, 'trial_id': str, 'cycle_id': str})
            source['model'] = 'selected_deep'
            source.to_csv(outer_dir / 'selected_deep.csv', index=False)
            write_json(outer_dir / 'winner.json', {'family': winner['family'], 'inner_subject_mae': winner['inner_subject_mae'],
                'refit': str((outer_dir / winner['family'] / 'refit').relative_to(root)),
                'rule': cfg['architecture_rule'], 'outer_subject': outer})
        write_json(outer_dir / 'complete.json', {'families': cfg['families'], 'outer_subject': outer})
    tables = [pd.read_csv(p, dtype={'participant_id': str, 'trial_id': str, 'cycle_id': str})
              for p in sorted((root / 'nested').glob('*/*/cycles.csv'))]
    tables += [pd.read_csv(p, dtype={'participant_id': str, 'trial_id': str, 'cycle_id': str})
               for p in sorted((root / 'nested').glob('*/selected_deep.csv'))]
    save_reports(pd.concat(tables, ignore_index=True), root / 'internal_reports', cfg)
    complete = [p.parent.name for p in (root / 'nested').glob('*/complete.json')]
    write_json(root / 'nested_coverage.json', {'available_subjects': ds.subjects, 'completed_subjects': sorted(complete),
                                             'complete': set(complete) == set(ds.subjects)})
    return root

def artifact_hashes(directory):
    return {str(p.relative_to(directory)): digest_file(p) for p in Path(directory).rglob('*') if p.is_file()}

def final_refit(cohort, cfg, root):
    root = Path(root)
    ds = coordinate_dataset(cohort, cfg['coordinate_mode'])
    bind_experiment(root, ds, cfg)
    if cfg['smoke']:
        raise ValueError('A smoke test cannot produce a research external-evaluation bundle')
    if set(cfg['families']) != set(BASELINES + DEEP):
        raise ValueError('Final release requires all four baselines and all three deep families')
    coverage = read_json(root / 'nested_coverage.json')
    if not coverage['complete'] or set(coverage['completed_subjects']) != set(ds.subjects):
        raise ValueError('Complete every internal outer participant before the final refit')
    freeze_path = root / 'frozen_external_plan.json'
    if freeze_path.exists():
        check_frozen(root)
        print('Frozen full-cohort models already exist; no retraining.', flush=True)
        return root
    selections = [tune_family(ds, family, cfg, root / 'full_internal_selection' / family) for family in cfg['families']]
    winner = choose_deep(selections)
    write_json(root / 'full_internal_selection' / 'architecture_selection.json',
               {'family': winner['family'], 'inner_subject_mae': winner['inner_subject_mae'],
                'note': 'Training selection score, not an unbiased performance estimate; use nested outer results for that.'})
    model_dirs = {}
    for chosen in selections:
        if chosen['family'] in DEEP and chosen['family'] != winner['family']:
            continue
        label = 'selected_deep' if chosen['family'] in DEEP else chosen['family']
        directory = root / 'final_models' / label
        fit_frozen_model(ds, chosen, cfg, directory)
        model_dirs[label] = str(directory.relative_to(root))
    ensemble = []
    if cfg['include_loso_ensemble']:
        ensemble = [read_json(root / 'nested' / s / 'winner.json')['refit'] for s in ds.subjects]
    files = artifact_hashes(root / 'final_models')
    hashes = {str(Path('final_models') / k): v for k, v in files.items()}
    for member in ensemble:
        hashes.update({str(Path(member) / k): v for k, v in artifact_hashes(root / member).items()})
    # External data/labels are never arguments to this function.
    frozen = {'created_utc': datetime.now(timezone.utc).isoformat(), 'config': cfg,
        'internal_context': cohort.context, 'internal_subjects': ds.subjects,
        'internal_global_ids': cohort.participants.get('global_subject_id', pd.Series(dtype=str)).replace('', np.nan).dropna().tolist(),
        'internal_fingerprint': ds.fingerprint, 'task': ds.task, 'model_directories': model_dirs,
        'selected_architecture': winner['family'], 'ensemble_members': ensemble, 'model_hashes': hashes,
        'source_hashes': source_hashes(), 'software': software(),
        'primary_model': 'selected_deep', 'inference_unit': 'participant; never model-times-participant',
        'external_status': cfg['external_status']}
    write_json(freeze_path, frozen)
    return root

def check_frozen(root):
    root = Path(root)
    frozen = read_json(root / 'frozen_external_plan.json')
    for name, expected in frozen['model_hashes'].items():
        if digest_file(root / name) != expected:
            raise ValueError('Frozen model changed: ' + name)
    if source_hashes() != frozen['source_hashes'] or software() != frozen['software']:
        raise ValueError('Source/environment changed since the model freeze')
    return frozen

def predict_frozen(root, frozen, x):
    output = {name: predict_saved(Path(root) / folder, x) for name, folder in frozen['model_directories'].items()}
    if frozen['ensemble_members']:
        output['loso_ensemble'] = mean_ensemble([predict_saved(Path(root) / p, x) for p in frozen['ensemble_members']])
    return output

def external_evaluation(root, external_directory):
    root = Path(root)
    frozen = check_frozen(root)  # Must exist BEFORE external files are opened.
    cfg = frozen['config']
    cohort = load_cohort(external_directory, frozen['task'], cfg['external_visits'], role='external')
    if not cohort.context.get('participant_disjointness_verified', False):
        raise ValueError('Verify that external participants are disjoint from the development cohort')
    external_global = cohort.participants.get('global_subject_id', pd.Series(dtype=str)).replace('', np.nan).dropna().tolist()
    if set(external_global) & set(frozen['internal_global_ids']):
        raise ValueError('Internal/external global participant identity overlap')
    if not cohort.context.get('units_verified', False) or not cohort.context.get('harmonization_verified', False):
        raise ValueError('Verify units, axis signs, gait events, and internal/external harmonization before evaluation')
    input_cohort = harmonize_input_units(cohort, frozen['internal_context']['ankle_units'])
    ds = coordinate_dataset(input_cohort, cfg['coordinate_mode'])
    output = root / 'external_evaluation'
    output.mkdir(parents=True, exist_ok=True)
    identity = {'frozen_plan_sha256': digest_file(root / 'frozen_external_plan.json'),
                'external_fingerprint': ds.fingerprint}
    lock = output / 'evaluation_identity.json'
    if lock.exists() and read_json(lock) != identity:
        raise ValueError('External evaluation was already bound to different data/models. Do not retune or replace the holdout.')
    if not lock.exists():
        write_json(lock, identity)
    if (output / 'complete.json').exists():
        if digest_file(output / 'predictions.npz') != read_json(output / 'complete.json')['prediction_sha256']:
            raise ValueError('Saved external predictions changed')
        print('External evaluation already completed. Returning the saved report without another prediction pass.', flush=True)
        return output
    audit_external(cohort, cfg['external_visits'], output / 'domain_shift_audit', frozen['internal_context'])
    cache = output / 'predictions.npz'
    if cache.exists():
        with np.load(cache, allow_pickle=False) as f:
            predictions = {name: f[name] for name in frozen['model_directories']}
            if frozen['ensemble_members']:
                predictions['loso_ensemble'] = f['loso_ensemble']
    else:
        predictions = predict_frozen(root, frozen, ds.x)
        temporary = output / 'predictions.tmp.npz'
        np.savez_compressed(temporary, **predictions, measured=ds.y,
                            cycle_ids=ds.cycles.cycle_id.to_numpy(str), participant_ids=ds.groups)
        temporary.replace(cache)
    tables = [prediction_table(ds, p, name) for name, p in predictions.items()]
    save_reports(pd.concat(tables, ignore_index=True), output, cfg, cohort.participants)
    write_json(output / 'interpretation.json', {'evaluation_type': 'domain-shift evaluation',
        'external_status': cfg['external_status'], 'primary_model': 'selected_deep',
        'prior_examination_note': 'Previously reported data cannot be relabeled untouched by rerunning a new protocol.'
           if cfg['external_status'] == 'previously_evaluated' else 'Untouched status is a user provenance declaration, not provable by code.',
        'participant_ci': 'Resamples whole trials within participants; unavailable with fewer than two known trials.',
        'cohort_ci': 'Resamples participants; visits remain separate and fitted-model uncertainty is not included.',
        'strata': 'Descriptive, prespecified, suppressed below minimum sample size; no causal explanation of domain shift.',
        'ensemble_rule': cfg['ensemble_rule'] if frozen['ensemble_members'] else 'No ensemble; one full-internal-cohort deep model',
        'external_adaptation': 'none; no external tuning, early stopping, or calibration'})
    write_json(output / 'complete.json', {**identity, 'prediction_sha256': digest_file(cache), 'n_unique_participants': len(ds.subjects)})
    return output

def compare_coordinates(experiment_root, modes, directory, cfg):
    """Paired comparisons on identical held-out participant/cycle membership."""
    rows, cohort_rows = [], []
    reference_cycles = None
    for mode in modes:
        root = Path(experiment_root) / mode
        if not read_json(root / 'nested_coverage.json')['complete']:
            raise ValueError(f'Incomplete outer coverage for {mode}')
        cycles = pd.read_csv(root / 'internal_reports' / 'cycle_metrics.csv', dtype={'participant_id': str, 'cycle_id': str})
        cycles = cycles[cycles.model == 'selected_deep']
        membership = set(zip(cycles.participant_id, cycles.cycle_id, cycles.visit))
        if reference_cycles is not None and membership != reference_cycles:
            raise ValueError('Coordinate comparisons require the same valid participant/cycle set')
        reference_cycles = membership
        table = pd.read_csv(root / 'internal_reports' / 'participant_metrics.csv', dtype={'participant_id': str})
        table = table[table.model == 'selected_deep'].copy()
        table['coordinate_mode'] = mode
        rows.append(table)
    all_rows = pd.concat(rows, ignore_index=True)
    if 'absolute' not in modes:
        raise ValueError('Include absolute coordinates as the reference')
    ref = all_rows[all_rows.coordinate_mode == 'absolute']
    for mode in modes:
        if mode == 'absolute':
            continue
        joined = all_rows[all_rows.coordinate_mode == mode].merge(ref, on=['task', 'participant_id', 'visit'], suffixes=('_mode', '_absolute'), validate='one_to_one')
        for metric in ['rmse', 'nrmse', 'mae']:
            delta = (joined[f'{metric}_absolute'] - joined[f'{metric}_mode']).dropna().to_numpy()
            lo, hi = bootstrap_interval(delta, stable_seed(cfg['bootstrap_seed'], mode, metric), cfg['bootstrap_resamples'])
            cohort_rows.append(dict(coordinate_mode=mode, metric=metric, n_participants=len(delta),
                median_improvement=float(np.median(delta)) if len(delta) else np.nan,
                median_improvement_ci95_low=lo, median_improvement_ci95_high=hi,
                interpretation='positive favors normalized coordinates; descriptive paired participant bootstrap'))
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    all_rows.to_csv(directory / 'participant_coordinate_metrics.csv', index=False)
    pd.DataFrame(cohort_rows).to_csv(directory / 'paired_coordinate_comparisons.csv', index=False)
    return directory

def perturb_internal(cohort, cfg, root):
    """Prespecified input nuisance tests on held-out INTERNAL cycles; no refitting."""
    root = Path(root)
    ds = coordinate_dataset(cohort, cfg['coordinate_mode'])
    bind_experiment(root, ds, cfg)
    if not cfg['run_perturbations']:
        raise ValueError('Enable perturbations in the prespecification before running the study')
    if not read_json(root / 'nested_coverage.json')['complete']:
        raise ValueError('Finish internal outer folds first')
    scenarios = [('unperturbed', None, 0.)]
    scenarios += [(f'translation_{i}', t, 0.) for i, t in enumerate(cfg['translation_offsets_m'])]
    if 'ankle_xyz' in cohort.arrays:
        scenarios += [(f'yaw_{angle:g}', None, angle) for angle in cfg['rotation_angles_deg']]
    elif cfg['rotation_angles_deg']:
        raise ValueError('Rotation tests require verified 3D data; do not rotate a scalar trace')
    rows = []
    for name, translation, angle in scenarios:
        transformed = coordinate_dataset(cohort, cfg['coordinate_mode'], translation, angle)
        for outer in ds.subjects:
            te = transformed.subset([outer])
            winner = read_json(root / 'nested' / outer / 'winner.json')
            prediction = predict_saved(root / winner['refit'], te.x)
            table = prediction_table(te, prediction, 'selected_deep')
            table['scenario'] = name
            rows.append(table)
    cycles = pd.concat(rows, ignore_index=True)
    output = root / 'perturbations'
    output.mkdir(parents=True, exist_ok=True)
    cycles.to_csv(output / 'cycle_metrics.csv', index=False)
    subjects = cycles.groupby(['task', 'model', 'participant_id', 'scenario'])[['rmse', 'nrmse', 'mae']].mean().reset_index()
    reference = subjects[subjects.scenario == 'unperturbed']
    delta = subjects.merge(reference.drop(columns='scenario'), on=['task', 'model', 'participant_id'], suffixes=('', '_unperturbed'), validate='many_to_one')
    for metric in ['rmse', 'nrmse', 'mae']:
        delta[f'{metric}_change'] = delta[metric] - delta[f'{metric}_unperturbed']
    delta.to_csv(output / 'participant_error_changes.csv', index=False)
    write_json(output / 'protocol.json', {'scenarios': scenarios,
        'definition': 'Rigid nuisance transform of ankle and pelvis positions together; direction rotates with them; gait-frame targets fixed.',
        'interpretation': 'Sensitivity evidence, not proof of biomechanical causality. No augmentation or parameter tuning on these tests.'})
    return output

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--stage', choices=['nested', 'final', 'external', 'perturb'], required=True)
    parser.add_argument('--config', required=True)
    parser.add_argument('--output', required=True)
    parser.add_argument('--task', required=True)
    parser.add_argument('--internal-zip')
    parser.add_argument('--internal-bundle')
    parser.add_argument('--external-bundle')
    parser.add_argument('--outer-subjects', nargs='+')
    args = parser.parse_args()
    cfg = read_json(args.config)
    if args.stage == 'external':
        if not args.external_bundle:
            parser.error('--external-bundle is required')
        frozen = check_frozen(args.output)
        if frozen['config'] != cfg or frozen['task'] != args.task:
            raise ValueError('External settings differ from the frozen protocol')
        external_evaluation(args.output, args.external_bundle)
        return
    if bool(args.internal_zip) == bool(args.internal_bundle):
        parser.error('Provide exactly one of --internal-zip or --internal-bundle')
    cohort = legacy_cohort(args.internal_zip, args.task) if args.internal_zip else load_cohort(args.internal_bundle, args.task, role='internal')
    if cfg['smoke']:
        cohort = replace(cohort, ds=cohort.ds.subset(cohort.ds.subjects[:3])) if not cohort.arrays else cohort
        if cohort.arrays:
            raise ValueError('Smoke test uses the original scalar ZIP; use normal mode for enriched bundles')
    if set(cfg['families']) & set(DEEP):
        from models import configure_tensorflow
        configure_tensorflow(cfg['seed'])
    if args.stage == 'nested':
        run_nested(cohort, cfg, args.output, args.outer_subjects or (cohort.ds.subjects[:1] if cfg['smoke'] else None))
    elif args.stage == 'final':
        final_refit(cohort, cfg, args.output)
    else:
        perturb_internal(cohort, cfg, args.output)

if __name__ == '__main__':
    main()
''')


In [ ]:
#@title Extension protocol tests
write_module('tests/test_revision.py', r'''import json
from dataclasses import replace
from pathlib import Path
import tempfile
import unittest
from unittest.mock import patch
import numpy as np
import pandas as pd
from data import Dataset
from protocol import read_json, write_json
from revision_data import Cohort, coordinate_dataset, load_cohort, REQUIRED_CONTEXT, harmonize_input_units
from revision_models import (participant_mean, fit_baseline, predict_baseline, mean_ensemble,
                             cv_candidate, BASELINES, DEEP)
from revision_reports import prediction_table, participant_table, save_reports, audit_external
import revision_study as study


def sample_cohort():
    rng = np.random.default_rng(5)
    groups = np.repeat(['P01', 'P02', 'P03'], 3)
    x = rng.normal(size=(9, 100, 1)).astype('float32')
    y = (0.2 * x + np.sin(np.linspace(0, 2 * np.pi, 100))[None, :, None]).astype('float32')
    meta = pd.DataFrame({'cycle_id': [f'C{i}' for i in range(9)], 'subject': groups, 'participant_id': groups,
        'trial_id': ['T0', 'T1', 'T1'] * 3, 'visit': 'internal', 'side': 'left', 'include': True, 'exclusion_reason': ''})
    ds = Dataset('AP_Left', x, y, groups, meta, 'synthetic-fixture')
    people = pd.DataFrame({'participant_id': ['P01', 'P02', 'P03'], 'sex': ['F', 'M', 'F'], 'age': [60, 70, 80]})
    context = {k: 'unknown' for k in REQUIRED_CONTEXT}
    context.update(cohort_id='test', dataset_role='internal', ankle_units='m', grf_units='N/kg',
                   units_verified=True, body_scale_definition='measured height in m')
    return Cohort(ds, context, people, pd.DataFrame(), meta, {})


def make_external_bundle(root):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    c = sample_cohort()
    context = dict(c.context, cohort_id='external_fixture', dataset_role='external', task='AP_Left',
                   harmonization_verified=True, participant_disjointness_verified=True)
    write_json(root / 'cohort.json', context)
    c.participants.to_csv(root / 'participants.csv', index=False)
    meta = c.ledger.copy()
    meta['visit'] = ['baseline', 'baseline', 'post_intervention'] * 3
    # T1 at baseline and T1 post are different trials, keyed by visit.
    trials = meta[['participant_id', 'trial_id', 'visit', 'side', 'include', 'exclusion_reason']].drop_duplicates()
    excluded = dict(participant_id='P01', trial_id='T99', visit='baseline', side='left', include=False, exclusion_reason='no force-plate contact')
    trials = pd.concat([trials, pd.DataFrame([excluded])], ignore_index=True)
    extra = {**excluded, 'cycle_id': 'excluded_cycle', 'subject': 'P01'}
    ledger = pd.concat([meta, pd.DataFrame([extra])], ignore_index=True)
    trials.to_csv(root / 'trials.csv', index=False)
    ledger.to_csv(root / 'cycles.csv', index=False)
    # Reverse waveform order to verify explicit IDs, never positional metadata pairing.
    np.savez_compressed(root / 'waveforms.npz', cycle_ids=meta.cycle_id.to_numpy(str)[::-1], x=c.ds.x[::-1], y=c.ds.y[::-1])
    return root


class RevisionTests(unittest.TestCase):
    def test_participant_weighted_mean_and_phase_ignore_ankle(self):
        y = np.zeros((4, 100, 1)); y[-1] = 10
        groups = np.array(['A', 'A', 'A', 'B'])
        self.assertTrue(np.allclose(participant_mean(y, groups), 5))
        x = np.zeros_like(y)
        for family, p in [('mean_waveform', {}), ('phase_only', {'harmonics': 3})]:
            model = fit_baseline(family, x, y, groups, p, 42)
            np.testing.assert_allclose(predict_baseline(model, x), predict_baseline(model, x + 1000))

    def test_mlp_is_single_hidden_layer_without_random_validation(self):
        c = sample_cohort()
        m = fit_baseline('shallow_mlp', c.ds.x, c.ds.y, c.ds.groups, {'units': 4, 'alpha': .01, 'max_iter': 5}, 42)
        self.assertFalse(m['model'].early_stopping)
        self.assertEqual(m['model'].hidden_layer_sizes, (4,))
        self.assertEqual(predict_baseline(m, c.ds.x).shape, c.ds.y.shape)

    def test_cv_separates_subjects_and_uses_equal_subject_score(self):
        c = sample_cohort(); ds = c.ds.subset(['P01', 'P02'])
        cfg = study.default_config()
        with tempfile.TemporaryDirectory() as tmp:
            result = cv_candidate(ds, 'ridge', {'alpha': 1}, cfg, tmp)
            for fold in result['folds']:
                self.assertNotIn(fold['validation_subject'], fold['training_subjects'])
                self.assertNotIn('P03', fold['training_subjects'])
            self.assertAlmostEqual(result['inner_subject_mae'], np.mean([f['mae'] for f in result['folds']]))

    def test_scalar_coordinates_cannot_fake_pelvis_or_rotation(self):
        c = sample_cohort()
        for mode in ['pelvis', 'combined', 'aligned', 'body']:
            with self.assertRaises(ValueError):
                coordinate_dataset(c, mode)
        with self.assertRaises(ValueError):
            coordinate_dataset(c, 'absolute', rotation_deg=15)
        a = coordinate_dataset(c, 'translation')
        b = coordinate_dataset(c, 'translation', [1, 0, 0])
        np.testing.assert_allclose(a.x, b.x, atol=1e-6)

    def test_physical_unit_conversion_does_not_fit_external_distribution(self):
        c = sample_cohort()
        converted = harmonize_input_units(c, 'mm')
        np.testing.assert_allclose(converted.ds.x, 1000 * c.ds.x)
        np.testing.assert_array_equal(converted.ds.y, c.ds.y)
        self.assertEqual(converted.context['ankle_units'], 'mm')

    def test_3d_coordinate_variants_keep_same_gait_force_target(self):
        with tempfile.TemporaryDirectory() as tmp:
            root = make_external_bundle(tmp)
            context = read_json(root / 'cohort.json')
            context['canonical_axis_order'] = ['forward', 'left', 'up']
            write_json(root / 'cohort.json', context)
            with np.load(root / 'waveforms.npz') as f:
                ids, x = f['cycle_ids'], f['x']
            xyz = np.repeat(x, 3, axis=2)
            forces = np.broadcast_to(np.array([1., 2., 3.]), xyz.shape)
            np.savez_compressed(root / 'waveforms.npz', cycle_ids=ids, ankle_xyz=xyz, grf_xyz=forces,
                direction_xyz=np.tile([0., 1., 0.], (len(x), 1)), pelvis_xyz=xyz / 2,
                body_scale_m=np.repeat(1.7, len(x)))
            c = load_cohort(root, 'AP_Left', ['baseline'])
            for mode in ['absolute', 'translation', 'pelvis', 'aligned', 'body', 'combined']:
                np.testing.assert_allclose(coordinate_dataset(c, mode).y, 2.)

    def test_combined_coordinates_are_translation_and_yaw_invariant(self):
        c = sample_cohort(); rng = np.random.default_rng(99)
        xyz = rng.normal(size=(9, 100, 3)); pelvis = rng.normal(size=xyz.shape)
        c = replace(c, arrays={'ankle_xyz': xyz, 'pelvis_xyz': pelvis, 'direction_xyz': np.tile([1., 0., 0.], (9, 1)), 'body_scale_m': np.repeat(1.7, 9)})
        original = coordinate_dataset(c, 'combined')
        nuisance = coordinate_dataset(c, 'combined', [1.2, -2.3, .5], 35)
        np.testing.assert_allclose(original.x, nuisance.x, atol=1e-6)
        np.testing.assert_array_equal(original.y, nuisance.y)

    def test_ensemble_averages_before_error(self):
        a, b = np.ones((2, 100, 1)), -np.ones((2, 100, 1))
        np.testing.assert_array_equal(mean_ensemble([a, b]), np.zeros_like(a))
        with self.assertRaises(ValueError):
            mean_ensemble([a, b[:1]])

    def test_trial_intervals_and_duplicate_model_rows(self):
        c = sample_cohort()
        cycles = prediction_table(c.ds, np.zeros_like(c.ds.y), 'selected_deep')
        rows = participant_table(cycles, repeats=100)
        self.assertEqual(len(rows), 3)
        self.assertTrue(rows.rmse_ci95_low.notna().all())
        cycles['trial_id'] = 'single_trial'
        rows = participant_table(cycles, repeats=100)
        self.assertTrue(rows.rmse_ci95_low.isna().all())
        with self.assertRaises(ValueError):
            participant_table(pd.concat([cycles, cycles]), repeats=100)

    def test_external_contract_preserves_visits_and_exclusion_counts(self):
        with tempfile.TemporaryDirectory() as tmp:
            root = make_external_bundle(Path(tmp) / 'data')
            c = load_cohort(root, 'AP_Left', ['baseline'], role='external')
            self.assertEqual(len(c.ds.x), 6)
            self.assertEqual(c.ds.cycles.cycle_id.iloc[0], 'C7')
            audit_external(c, ['baseline'], Path(tmp) / 'audit', {})
            flow = read_json(Path(tmp) / 'audit' / 'cohort_flow_totals.json')
            self.assertEqual(flow['n_valid_cycles'], 6)
            self.assertEqual(flow['n_trials_recorded'], 10)
            excluded = pd.read_csv(Path(tmp) / 'audit' / 'cycle_exclusion_reasons.csv')
            self.assertIn('no force-plate contact', excluded.analysis_exclusion_reason.tolist())

    def test_nested_baseline_integration_and_resume(self):
        c = sample_cohort(); cfg = study.default_config()
        cfg.update(families=list(BASELINES), smoke=True, bootstrap_resamples=100)
        with tempfile.TemporaryDirectory() as tmp:
            study.run_nested(c, cfg, tmp)
            self.assertTrue(read_json(Path(tmp) / 'nested_coverage.json')['complete'])
            rows = pd.read_csv(Path(tmp) / 'internal_reports' / 'participant_metrics.csv')
            self.assertEqual(len(rows), 12)
            with patch('revision_study.predict_saved', side_effect=AssertionError('Should resume')):
                study.run_nested(c, cfg, tmp)

    def test_full_refit_freezes_internal_only_and_external_runs_once(self):
        c = sample_cohort(); cfg = study.default_config(); cfg['bootstrap_resamples'] = 100
        cfg['include_loso_ensemble'] = True
        trained_subjects = []
        def fake_tune(ds, family, cfg, directory):
            scores = {'lstm': 3., 'gru': 1., 'conv1d_lstm': 2.}
            return dict(family=family, parameters={}, epochs=1, inner_subject_mae=scores.get(family, 4.), folds=[])
        def fake_fit(ds, selected, cfg, directory):
            # Recording test double, saved as a real mean model for inference tests.
            trained_subjects.append(tuple(ds.subjects))
            actual = dict(family='mean_waveform', parameters={}, epochs=None)
            return real_fit(ds, actual, cfg, directory)
        real_fit = study.fit_frozen_model
        with tempfile.TemporaryDirectory() as tmp:
            root = Path(tmp) / 'experiment'
            with patch('revision_study.tune_family', fake_tune), patch('revision_study.fit_frozen_model', fake_fit):
                study.run_nested(c, cfg, root)
                study.final_refit(c, cfg, root)
            self.assertEqual(trained_subjects[-5:], [tuple(c.ds.subjects)] * 5)
            frozen = read_json(root / 'frozen_external_plan.json')
            self.assertEqual(frozen['selected_architecture'], 'gru')
            self.assertEqual(len(frozen['ensemble_members']), 3)
            external = make_external_bundle(Path(tmp) / 'external')
            output = study.external_evaluation(root, external)
            rows = pd.read_csv(output / 'participant_metrics.csv')
            self.assertEqual(len(rows), 3 * 6)  # Three people, six predictors; not 18 people.
            with patch('revision_study.predict_frozen', side_effect=AssertionError('Second external pass')):
                study.external_evaluation(root, external)
            frozen['config']['external_visits'] = ['post_intervention']
            write_json(root / 'frozen_external_plan.json', frozen)
            with self.assertRaises(ValueError):
                study.external_evaluation(root, external)

    def test_final_model_rejects_incomplete_outer_validation(self):
        c = sample_cohort(); cfg = study.default_config()
        with tempfile.TemporaryDirectory() as tmp:
            study.bind_experiment(tmp, coordinate_dataset(c, cfg['coordinate_mode']), cfg)
            write_json(Path(tmp) / 'nested_coverage.json', {'complete': False, 'completed_subjects': ['P01']})
            with self.assertRaises(ValueError):
                study.final_refit(c, cfg, tmp)

if __name__ == '__main__':
    unittest.main()
''')


In [ ]:
#@title external_workbook.py
write_module('external_workbook.py', r'''"""Read the supplied anonymous external workbook without inventing identities."""
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from protocol import write_json, read_json
from revision_data import load_cohort, bool_column, digest_file

SHEETS = {f'{component}_{side}': (f'{side[0]}-{force}', f'{side[0]}ANK')
          for side in ['Left', 'Right']
          for component, force in [('AP', 'AP'), ('Vertical', 'VERT'), ('ML', 'ML')]}

def audit_workbook(path):
    tables = pd.read_excel(path, sheet_name=None, header=None, engine='openpyxl')
    rows = []
    for name, frame in tables.items():
        if frame.empty:
            rows.append(dict(sheet=name, samples=0, columns=0, missing_values=0, status='empty'))
            continue
        try:
            x = frame.to_numpy(dtype=float)
        except ValueError:
            raise ValueError(f'{name}: expected an all-numeric sheet with no header row')
        rows.append(dict(sheet=name, samples=x.shape[0], columns=x.shape[1],
                         missing_values=int((~np.isfinite(x)).sum()), minimum=float(np.nanmin(x)),
                         maximum=float(np.nanmax(x)), status='numeric waveform sheet'))
    return tables, pd.DataFrame(rows)

def make_mapping_templates(workbook, directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    tables, audit = audit_workbook(workbook)
    audit.to_csv(directory / 'external_workbook_audit.csv', index=False)
    mapping = []
    for side in ['Left', 'Right']:
        frame = tables[f'{side[0]}-VERT']
        for column in range(1, frame.shape[1] + 1):
            mapping.append(dict(side=side.lower(), column_1based=column,
                cycle_id=f'EXT_{side.upper()}_COL{column:03d}', participant_id='', trial_id='',
                visit='', include='', exclusion_reason='', body_mass_kg='',
                age='', sex='', medication_state='', HY_stage='', intervention_group='', global_subject_id=''))
    # Never overwrite metadata the user has filled in.
    if not (directory / 'external_column_map.csv').exists():
        pd.DataFrame(mapping).to_csv(directory / 'external_column_map.csv', index=False)
    spec = dict(workbook_sha256=digest_file(workbook), ankle_units='UNCONFIRMED', force_units='UNCONFIRMED',
        ankle_axis={'AP': 'UNCONFIRMED', 'Vertical': 'UNCONFIRMED', 'ML': 'UNCONFIRMED'},
        ankle_sign={'AP': None, 'Vertical': None, 'ML': None},
        force_sign={'AP': None, 'Vertical': None, 'ML': None},
        phase_grid_0_to_100_confirmed=False, column_pairing_verified=False,
        internal_external_axes_and_signs_verified=False, participant_disjointness_verified=False,
        visit_mapping_verified=False, identity_mapping_verified=False,
        coordinate_definition='UNCONFIRMED', motion_capture_system='unknown', force_platforms='unknown',
        kinematics_sampling_hz='unknown', force_sampling_hz='unknown', walking_protocol='unknown',
        cycle_definition='UNCONFIRMED', harmonization_description='UNCONFIRMED',
        resampling='linear 101-to-100 normalized-phase interpolation; preserve both endpoints',
        source_population_scope='Provided workbook columns only; source recruitment/exclusion history unavailable')
    if not (directory / 'external_import_settings.json').exists():
        write_json(directory / 'external_import_settings.json', spec)
    return audit

def readiness(workbook, directory):
    directory = Path(directory)
    reasons = []
    needed = ['external_column_map.csv', 'external_import_settings.json']
    if any(not (directory / name).exists() for name in needed):
        return ['External column map or import settings not supplied']
    spec = read_json(directory / needed[1])
    if spec.get('workbook_sha256') != digest_file(workbook):
        reasons.append('External mapping/settings refer to a different workbook')
    if spec.get('ankle_units') not in ('mm', 'm'):
        reasons.append('Confirm ankle units: mm or m')
    if spec.get('force_units') not in ('N', 'N/kg'):
        reasons.append('Confirm force units: N or N/kg')
    for flag in ['phase_grid_0_to_100_confirmed', 'column_pairing_verified',
                 'internal_external_axes_and_signs_verified', 'participant_disjointness_verified',
                 'visit_mapping_verified', 'identity_mapping_verified']:
        if spec.get(flag) is not True:
            reasons.append('Source verification needed: ' + flag)
    for component in ['AP', 'Vertical', 'ML']:
        if spec.get('ankle_axis', {}).get(component) not in ('X', 'Y', 'Z'):
            reasons.append('Confirm ankle axis for ' + component)
        for item in ['ankle_sign', 'force_sign']:
            if spec.get(item, {}).get(component) not in (-1, 1):
                reasons.append(f'Confirm {item} for {component}: +1 or -1')
    for field in ['coordinate_definition', 'cycle_definition', 'harmonization_description']:
        if spec.get(field) in (None, '', 'UNCONFIRMED'):
            reasons.append('Complete ' + field)
    mapping = pd.read_csv(directory / needed[0], dtype=str, keep_default_na=False)
    fields = ['participant_id', 'trial_id', 'visit', 'include']
    for field in fields:
        if field not in mapping or mapping[field].str.strip().eq('').any():
            reasons.append('Complete external_column_map.csv: ' + field)
    if not reasons:
        included = bool_column(mapping.include)
        if not included.any():
            reasons.append('No included external waveform columns')
        if spec['force_units'] == 'N':
            mass = pd.to_numeric(mapping.loc[included, 'body_mass_kg'], errors='coerce')
            if mass.isna().any() or (mass <= 0).any():
                reasons.append('Positive source body_mass_kg is required to convert N to N/kg')
    return reasons

def prepare_external_bundles(workbook, metadata_dir, output_root):
    """Deterministic import; called after internal model freeze, never tunes using labels."""
    reasons = readiness(workbook, metadata_dir)
    if reasons:
        raise ValueError('; '.join(reasons))
    metadata_dir, output_root = Path(metadata_dir), Path(output_root)
    settings = read_json(metadata_dir / 'external_import_settings.json')
    mapping = pd.read_csv(metadata_dir / 'external_column_map.csv', dtype=str, keep_default_na=False)
    mapping['column_1based'] = pd.to_numeric(mapping.column_1based, errors='raise').astype(int)
    mapping['include'] = bool_column(mapping.include)
    if mapping.duplicated(['side', 'column_1based']).any() or mapping.cycle_id.duplicated().any():
        raise ValueError('Duplicate source column or cycle ID')
    tables, audit = audit_workbook(workbook)
    person_cols = ['age', 'sex', 'medication_state', 'HY_stage', 'intervention_group', 'global_subject_id']
    for _, group in mapping.groupby('participant_id'):
        for col in [*person_cols, 'body_mass_kg']:
            values = set(group[col]) - {''}
            if len(values) > 1:
                raise ValueError(f'{col} differs within a participant; use a visit-specific metadata design before import')
    people = mapping.groupby('participant_id', sort=True)[person_cols].agg(lambda s: next((x for x in s if x != ''), '')).reset_index()
    outputs = {}
    for task, (force_sheet, ankle_prefix) in SHEETS.items():
        component, side = task.split('_')
        rows = mapping[mapping.side == side.lower()].sort_values('column_1based').copy()
        n_columns = tables[force_sheet].shape[1]
        if set(rows.column_1based) != set(range(1, n_columns + 1)):
            raise ValueError(f'{side}: map every source column exactly once, including exclusions')
        force = tables[force_sheet].to_numpy(float)
        ankle = tables[ankle_prefix + settings['ankle_axis'][component]].to_numpy(float)
        if force.shape != (101, n_columns) or ankle.shape != force.shape:
            raise ValueError('Expected matched 101-sample column arrays; first row is data, not a header')
        kept = rows[rows.include]
        if kept.empty:
            continue
        idx = kept.column_1based.to_numpy() - 1
        x = ankle[:, idx].T * settings['ankle_sign'][component]
        y = force[:, idx].T * settings['force_sign'][component]
        if not np.isfinite(x).all() or not np.isfinite(y).all():
            raise ValueError('Missing/nonfinite included waveforms; document exclusions or repair source data')
        if settings['force_units'] == 'N':
            mass = pd.to_numeric(kept.body_mass_kg, errors='raise').to_numpy(float)
            y /= mass[:, None]
        phase_from, phase_to = np.linspace(0, 100, 101), np.linspace(0, 100, 100)
        x = np.stack([np.interp(phase_to, phase_from, v) for v in x])[:, :, None].astype('float32')
        y = np.stack([np.interp(phase_to, phase_from, v) for v in y])[:, :, None].astype('float32')
        folder = output_root / task
        folder.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(folder / 'waveforms.npz', x=x, y=y, cycle_ids=kept.cycle_id.to_numpy(str))
        rows.to_csv(folder / 'cycles.csv', index=False)
        # Trial status derived only for trials represented by workbook columns.
        trial_keys = ['participant_id', 'trial_id', 'visit', 'side']
        trials = rows.groupby(trial_keys).agg(include=('include', 'any'),
            exclusion_reason=('exclusion_reason', lambda s: '; '.join(sorted(set(s) - {''})))).reset_index()
        trials.loc[trials.include, 'exclusion_reason'] = ''
        trials.to_csv(folder / 'trials.csv', index=False)
        people.to_csv(folder / 'participants.csv', index=False)
        context = {key: settings[key] for key in ['ankle_units', 'coordinate_definition', 'motion_capture_system',
            'force_platforms', 'kinematics_sampling_hz', 'force_sampling_hz', 'walking_protocol',
            'cycle_definition', 'harmonization_description', 'source_population_scope']}
        context.update(cohort_id='Viglialoro2025_external_workbook', dataset_role='external', task=task,
            grf_units='N/kg', units_verified=True, harmonization_verified=True,
            participant_disjointness_verified=True, source_workbook_sha256=digest_file(workbook),
            import_settings_sha256=digest_file(metadata_dir / 'external_import_settings.json'),
            column_map_sha256=digest_file(metadata_dir / 'external_column_map.csv'),
            force_source_units=settings['force_units'], time_normalization=settings['resampling'],
            exclusions_scope='Only columns and represented trials in this workbook; source exclusions unavailable')
        write_json(folder / 'cohort.json', context)
        load_cohort(folder, task, role='external')
        outputs[task] = str(folder)
    return outputs
''')


In [ ]:
#@title complete_study.py
write_module('complete_study.py', r'''"""All-task study controller. Completed stages resume; missing data stay explicit."""
from pathlib import Path
from datetime import datetime, timezone
import argparse
import json
import traceback
import numpy as np
import pandas as pd
from data import TASK_ORDER
from protocol import write_json, read_json
from revision_data import legacy_cohort, load_cohort, coordinate_dataset, digest_file, MODES
from revision_models import BASELINES, DEEP, baseline_grid
from revision_study import (default_config, validate_config, run_nested, final_refit,
    perturb_internal, compare_coordinates, external_evaluation)
from external_workbook import readiness, prepare_external_bundles

def load_internal(plan, task):
    if plan['internal_source'] == 'original_zip':
        return legacy_cohort(plan['internal_zip'], task)
    if plan['internal_source'] == 'enriched_bundles':
        return load_cohort(Path(plan['internal_bundle_root']) / task, task, role='internal')
    raise ValueError('Unknown internal_source')

def configurations(plan, mode):
    cfg = {**default_config(), **plan['training'], 'smoke': False, 'coordinate_mode': mode}
    cfg['families'] = list(BASELINES + DEEP)
    validate_config(cfg)
    return cfg

def inspect_plan(plan):
    if not plan['tasks'] or len(set(plan['tasks'])) != len(plan['tasks']) or set(plan['tasks']) - set(TASK_ORDER):
        raise ValueError('Unknown/duplicate task')
    if not plan['coordinate_modes'] or set(plan['coordinate_modes']) - set(MODES):
        raise ValueError('Unknown coordinate mode')
    if plan['primary_coordinates'] not in plan['coordinate_modes']:
        raise ValueError('Primary coordinate mode must be prespecified in the requested modes')
    rows = []
    for task in plan['tasks']:
        try:
            cohort = load_internal(plan, task)
        except (ValueError, FileNotFoundError) as exc:
            rows.append(dict(task=task, coordinate='all', stage='input', status='blocked', reason=str(exc), fit_upper_bound=0))
            continue
        for mode in plan['coordinate_modes']:
            cfg = configurations(plan, mode)
            try:
                ds = coordinate_dataset(cohort, mode)
                n = len(ds.subjects)
                candidates = 3 * cfg['max_trials'] + sum(len(baseline_grid(f)) for f in BASELINES)
                fits = n * (candidates * (n - 1) + 7)
                rows.append(dict(task=task, coordinate=mode, stage='nested', status='ready', reason='',
                    n_participants=n, n_cycles=len(ds.x), fit_upper_bound=fits))
                if mode == plan['primary_coordinates']:
                    rows.append(dict(task=task, coordinate=mode, stage='final', status='after nested completion', reason='',
                        n_participants=n, n_cycles=len(ds.x), fit_upper_bound=n * candidates + 5))
                if cfg['run_perturbations']:
                    try:
                        for t in cfg['translation_offsets_m']:
                            coordinate_dataset(cohort, mode, translation_m=t)
                        for angle in cfg['rotation_angles_deg']:
                            coordinate_dataset(cohort, mode, rotation_deg=angle)
                        state, reason = 'after nested completion', ''
                    except ValueError as exc:
                        state, reason = 'blocked', str(exc)
                    rows.append(dict(task=task, coordinate=mode, stage='perturb', status=state, reason=reason, fit_upper_bound=0))
            except ValueError as exc:
                rows.append(dict(task=task, coordinate=mode, stage='nested', status='blocked', reason=str(exc), fit_upper_bound=0))
    return pd.DataFrame(rows)

def lock_plan(plan, root):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    # Runtime location of uploaded files can change; content is checked by the study
    # and external loaders. Scientific settings and adapter source must not change.
    signature = {k: plan[k] for k in ['tasks', 'coordinate_modes', 'primary_coordinates', 'internal_source', 'training']}
    signature['controller_sha256'] = digest_file(Path(__file__))
    signature['external_adapter_sha256'] = digest_file(Path(__file__).with_name('external_workbook.py'))
    path = root / 'complete_run_plan.json'
    if path.exists() and read_json(path) != signature:
        raise ValueError('The complete-run scientific plan changed. Restore it or use a new experiment name.')
    if not path.exists():
        write_json(path, signature)

def run_batch(plan):
    root = Path(plan['output_root'])
    lock_plan(plan, root)
    preflight = inspect_plan(plan)
    preflight.to_csv(root / 'preflight.csv', index=False)
    print(preflight[['task', 'coordinate', 'stage', 'status', 'fit_upper_bound']].to_string(index=False), flush=True)
    print(f'Total planned model-fit upper bound: {int(preflight.fit_upper_bound.sum()):,}', flush=True)
    states = read_json(root / 'progress.json') if (root / 'progress.json').exists() else {}

    def record(task, mode, stage, status, reason=''):
        key = f'{task}/{mode}/{stage}'
        states[key] = dict(task=task, coordinate=mode, stage=stage, status=status, reason=reason,
                           updated_utc=datetime.now(timezone.utc).isoformat())
        write_json(root / 'progress.json', states)
        pd.DataFrame(states.values()).to_csv(root / 'progress.csv', index=False)
        print(f'[{status.upper()}] {key}' + (f': {reason}' if reason else ''), flush=True)

    def execute(task, mode, stage, function):
        record(task, mode, stage, 'running')
        try:
            function()
        except Exception as exc:
            record(task, mode, stage, 'failed', str(exc))
            raise
        record(task, mode, stage, 'completed')

    for row in preflight.itertuples():
        if row.status == 'blocked':
            record(row.task, row.coordinate, row.stage, 'blocked', row.reason)
    if not (preflight.status == 'ready').any():
        print('No internal analyses are ready. Resolve the listed input issues.', flush=True)
        return
    from models import configure_tensorflow
    hardware = configure_tensorflow(plan['training']['seed'])
    write_json(root / 'runtime_hardware.json', hardware)
    for task in plan['tasks']:
        subset = preflight[preflight.task == task]
        ready_modes = subset.loc[(subset.stage == 'nested') & (subset.status == 'ready'), 'coordinate'].tolist()
        if not ready_modes:
            continue
        cohort = load_internal(plan, task)
        for mode in ready_modes:
            cfg = configurations(plan, mode)
            experiment = root / task / mode
            execute(task, mode, 'nested', lambda: run_nested(cohort, cfg, experiment))
            if ((subset.coordinate == mode) & (subset.stage == 'perturb') & (subset.status != 'blocked')).any():
                execute(task, mode, 'perturb', lambda: perturb_internal(cohort, cfg, experiment))
        if 'absolute' in ready_modes and len(ready_modes) > 1:
            execute(task, 'comparison', 'coordinates', lambda: compare_coordinates(root / task, ready_modes,
                root / task / 'coordinate_comparison', configurations(plan, 'absolute')))
        primary = plan['primary_coordinates']
        if primary not in ready_modes:
            record(task, primary, 'final', 'blocked', 'Prespecified primary coordinates unavailable; no automatic substitution')
            record(task, primary, 'external', 'blocked', 'Final primary model unavailable')
            continue
        cfg, experiment = configurations(plan, primary), root / task / primary
        execute(task, primary, 'final', lambda: final_refit(cohort, cfg, experiment))
        # Only NOW, after the final model freeze, import external targets.
        external_folder = Path(plan['external_bundle_root']) / task
        files = ['waveforms.npz', 'participants.csv', 'trials.csv', 'cycles.csv', 'cohort.json']
        if not all((external_folder / p).is_file() for p in files):
            if not plan.get('external_workbook') or not Path(plan['external_workbook']).is_file():
                record(task, primary, 'external', 'blocked', 'External workbook or prepared cohort bundle missing')
                continue
            reasons = readiness(plan['external_workbook'], plan['external_metadata_dir'])
            if reasons:
                record(task, primary, 'external', 'blocked', '; '.join(reasons))
                continue
            prepare_external_bundles(plan['external_workbook'], plan['external_metadata_dir'], plan['external_bundle_root'])
        context = read_json(external_folder / 'cohort.json')
        missing_flags = [name for name in ['units_verified', 'harmonization_verified', 'participant_disjointness_verified']
                         if context.get(name) is not True]
        if missing_flags:
            record(task, primary, 'external', 'blocked', 'Complete source verification: ' + ', '.join(missing_flags))
            continue
        execute(task, primary, 'external', lambda: external_evaluation(experiment, external_folder))
    frames = []
    for task in plan['tasks']:
        for mode in plan['coordinate_modes']:
            p = root / task / mode / 'internal_reports' / 'cohort_summary.csv'
            if p.exists():
                frame = pd.read_csv(p)
                frame['coordinate_mode'] = mode
                frames.append(frame)
    if frames:
        pd.concat(frames, ignore_index=True).to_csv(root / 'all_internal_cohort_summaries.csv', index=False)
    counts = pd.Series([r['status'] for r in states.values()]).value_counts().to_dict()
    write_json(root / 'completion_status.json', {'stage_counts': counts,
        'all_requested_stages_complete': all(r['status'] == 'completed' for r in states.values()),
        'note': 'Completed means the stage ran; blocked stages require the stated data. Missing source inputs are never imputed.'})
    print('Available work finished. Read progress.csv for completed and blocked stages.', flush=True)

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--plan', required=True)
    parser.add_argument('--preflight-only', action='store_true')
    args = parser.parse_args()
    plan = read_json(args.plan)
    if args.preflight_only:
        table = inspect_plan(plan)
        table.to_csv(Path(args.plan).with_name('preflight_preview.csv'), index=False)
        print(table.to_string(index=False))
        print('FIT UPPER BOUND:', int(table.fit_upper_bound.sum()))
    else:
        run_batch(plan)
''')


In [ ]:
#@title tests/test_complete_run.py
write_module('tests/test_complete_run.py', r'''import tempfile
import unittest
from pathlib import Path
from unittest.mock import patch
import numpy as np
import pandas as pd
from external_workbook import make_mapping_templates, readiness, prepare_external_bundles
from protocol import read_json, write_json
from revision_study import default_config
from test_revision import sample_cohort
from complete_study import inspect_plan, run_batch

class CompleteRunTests(unittest.TestCase):
    def tables(self):
        base = np.column_stack([np.linspace(10, 110, 101), np.linspace(20, 220, 101)])
        tables = {}
        for side in ['L', 'R']:
            for force in ['AP', 'VERT', 'ML']:
                tables[f'{side}-{force}'] = pd.DataFrame(-base * 10)
            for axis in ['X', 'Y', 'Z']:
                tables[f'{side}ANK{axis}'] = pd.DataFrame(base)
        return tables

    def test_external_unknown_identity_and_units_block_import(self):
        with tempfile.TemporaryDirectory() as tmp:
            root = Path(tmp); book = root / 'fixture.xlsx'; book.write_bytes(b'fixture')
            with patch('external_workbook.pd.read_excel', return_value=self.tables()):
                make_mapping_templates(book, root / 'metadata')
            why = readiness(book, root / 'metadata')
            self.assertTrue(any('participant_id' in s for s in why))
            self.assertTrue(any('force units' in s for s in why))
            with self.assertRaises(ValueError):
                prepare_external_bundles(book, root / 'metadata', root / 'bundles')

    def test_direct_import_uses_first_sample_and_physical_mass_conversion(self):
        with tempfile.TemporaryDirectory() as tmp:
            root = Path(tmp); book = root / 'fixture.xlsx'; book.write_bytes(b'fixture')
            meta = root / 'metadata'
            with patch('external_workbook.pd.read_excel', return_value=self.tables()):
                make_mapping_templates(book, meta)
            df = pd.read_csv(meta / 'external_column_map.csv', keep_default_na=False)
            df['participant_id'] = ['TEST01', 'TEST02', 'TEST01', 'TEST02']
            df['trial_id'] = 'trial1'; df['visit'] = 'baseline'; df['include'] = True
            df['body_mass_kg'] = 50
            df.to_csv(meta / 'external_column_map.csv', index=False)
            settings = read_json(meta / 'external_import_settings.json')
            for k in ['phase_grid_0_to_100_confirmed', 'column_pairing_verified',
                      'internal_external_axes_and_signs_verified', 'participant_disjointness_verified',
                      'visit_mapping_verified', 'identity_mapping_verified']:
                settings[k] = True
            settings.update(ankle_units='mm', force_units='N', ankle_axis={'AP':'Y','Vertical':'Z','ML':'X'},
                ankle_sign={'AP':1,'Vertical':1,'ML':1}, force_sign={'AP':-1,'Vertical':-1,'ML':-1},
                coordinate_definition='TEST fixture', cycle_definition='TEST full cycle', harmonization_description='TEST mapping')
            write_json(meta / 'external_import_settings.json', settings)
            with patch('external_workbook.pd.read_excel', return_value=self.tables()):
                result = prepare_external_bundles(book, meta, root / 'bundles')
            self.assertEqual(len(result), 6)
            with np.load(Path(result['AP_Left']) / 'waveforms.npz') as f:
                self.assertEqual(f['x'].shape, (2,100,1))
                self.assertAlmostEqual(float(f['x'][0,0,0]), 10)
                self.assertAlmostEqual(float(f['x'][0,-1,0]), 110)
                self.assertAlmostEqual(float(f['y'][0,0,0]), 2)

    def test_batch_runs_all_tasks_and_keeps_missing_stages_blocked(self):
        with tempfile.TemporaryDirectory() as tmp:
            cfg = default_config(); cfg['run_perturbations'] = True
            plan = dict(tasks=['AP_Left','AP_Right','Vertical_Left','Vertical_Right','ML_Left','ML_Right'],
                coordinate_modes=['absolute','translation','pelvis'], primary_coordinates='absolute',
                internal_source='original_zip', training=cfg, output_root=str(Path(tmp)/'study'),
                external_bundle_root=str(Path(tmp)/'external'), external_workbook='', external_metadata_dir='')
            calls = []
            def nested(cohort,cfg,root): calls.append(('nested',str(root)))
            def final(cohort,cfg,root): calls.append(('final',str(root)))
            # Scalar fixture can translate but cannot support pelvis or yaw.
            with patch('complete_study.load_internal', return_value=sample_cohort()), \
                 patch('complete_study.run_nested', side_effect=nested), \
                 patch('complete_study.final_refit', side_effect=final), \
                 patch('complete_study.compare_coordinates'), \
                 patch('models.configure_tensorflow', return_value={'test':'no real neural training'}):
                run_batch(plan)
            self.assertEqual(sum(t=='nested' for t,_ in calls), 12)
            self.assertEqual(sum(t=='final' for t,_ in calls), 6)
            progress = read_json(Path(tmp)/'study'/'progress.json')
            self.assertEqual(sum(v['stage']=='external' and v['status']=='blocked' for v in progress.values()), 6)
            self.assertFalse(read_json(Path(tmp)/'study'/'completion_status.json')['all_requested_stages_complete'])

if __name__ == '__main__':
    unittest.main()
''')


## 4 Verify the code, tests, and internal dataset

Synthetic test-fixture results are not cohort results. The study starts in section 7.


In [ ]:
#@title Validate the embedded files and initialize the command runner
import ast
EXPECTED_SOURCE_HASHES = {'data.py': 'c87ef77ed55f10d92abae2a8270aba57020fe8790c890249a3faf4f88df2465d', 'protocol.py': 'a4cfc95095c89ec482a800244cfa74a001e793c3b61a09837a0d9626ee2e9a34', 'models.py': '3f30671d985206c4fe1d5eaca53eef65f8b012ec01870e91db6e96a15627a258', 'nested_loso.py': '236f9083c47102e21c3b45461c17df3f46af00473be4b9b7caf056983041543c', 'tests/test_protocol.py': '5ac200393d5c14af18f90d007deba92e62e3e0d0108eb8c39bd0d0529bf8d448'}
for name, expected in EXPECTED_SOURCE_HASHES.items():
    actual = file_sha256(CODE_DIR / name)
    ast.parse((CODE_DIR / name).read_text(encoding="utf-8"))
    if actual != expected:
        raise ValueError(f"Embedded source changed: {name}. Use a new experiment and update provenance before research training.")

NOTEBOOK_CODE_HASH = hashlib.sha256(
    json.dumps(EXPECTED_SOURCE_HASHES, sort_keys=True).encode()).hexdigest()
snapshot_dir = PROJECT_DIR / "code_snapshots" / NOTEBOOK_CODE_HASH[:16]
snapshot_dir.mkdir(parents=True, exist_ok=True)
for name in EXPECTED_SOURCE_HASHES:
    destination = snapshot_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CODE_DIR / name, destination)

def run_python(arguments, label):
    log_dir = PROJECT_DIR / "notebook_logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    log_path = log_dir / f"{timestamp}_{label}.log"
    command = [sys.executable, "-u", *map(str, arguments)]
    environment = os.environ.copy()
    environment["PYTHONUNBUFFERED"] = "1"
    print("Starting:", label)
    process = subprocess.Popen(command, cwd=CODE_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1,
                               env=environment)
    try:
        with log_path.open("w", encoding="utf-8") as log:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
        return_code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("Interrupted. Completed fits remain saved. Rerun with the same settings and RESUME=True.")
        raise
    finally:
        process.stdout.close()
    if return_code:
        raise RuntimeError(f"{label} failed (exit {return_code}). Read the output above; log: {log_path}")
    return log_path

# The complete test suite runs in the next verification cell.
environment_dir = PROJECT_DIR / "environments"
environment_dir.mkdir(parents=True, exist_ok=True)
ENVIRONMENT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
(environment_dir / f"{ENVIRONMENT_ID}.json").write_text(
    json.dumps({"runtime": RUNTIME_INFO, "dataset_sha256": DATA_DIGEST,
                "notebook_code_hash": NOTEBOOK_CODE_HASH}, indent=2), encoding="utf-8")
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(environment_dir / f"{ENVIRONMENT_ID}_pip_freeze.txt").write_text(freeze, encoding="utf-8")
print("Source snapshot:", snapshot_dir)


In [ ]:
#@title Check the extension and save its source snapshot
EXTENSION_HASHES = {'revision_data.py': 'f8873fb162c58822a6486c2f8094ec469178fca171b87406b6bbe28eb18c5d1c', 'revision_models.py': 'e505b318db62817ecdcb1e07f2fedb2e369f2765c60ddd6fa345c6dfc51cb9f8', 'revision_reports.py': 'a757e7f5e1627650f1d8d8c974d129acf675655e57073fb5b58b5ba58676f5d7', 'revision_study.py': 'a5b88a16ed570718789fd095a5af0fb60a5cb4073f328a03c645d9a751cc4255', 'tests/test_revision.py': 'ccdefac38f94254bd19f9186dec8d4112dbbdd5968ab544fb4c7a2a9faa3894d'}
for name, expected in EXTENSION_HASHES.items():
    if file_sha256(CODE_DIR / name) != expected:
        raise ValueError(f'Extension source mismatch: {name}')
    ast.parse((CODE_DIR / name).read_text())
run_python(['-m', 'unittest', 'discover', '-s', 'tests', '-v'], 'all_protocol_tests')
extension_snapshot = PROJECT_DIR / 'extension_code' / hashlib.sha256(json.dumps(EXTENSION_HASHES, sort_keys=True).encode()).hexdigest()[:16]
for name in [*EXPECTED_SOURCE_HASHES, *EXTENSION_HASHES]:
    target = extension_snapshot / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CODE_DIR / name, target)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# Reload helpers if this notebook was edited in the current runtime.
for module in ['revision_study', 'revision_models', 'revision_reports', 'revision_data']:
    sys.modules.pop(module, None)
from revision_study import default_config
from revision_data import write_templates
print('Extension ready. Neural/GPU execution is checked by the smoke test, not these tests.')


In [ ]:
#@title Verify the complete-run source files
COMPLETE_SOURCE_HASHES = {'external_workbook.py': '37c0b3baed357559a789ec6b38b3b90043c8f7cf72908aa06e88bbadbacdf382', 'complete_study.py': '04c150a47bfaeec41349cad3aa7d2978af1382e791c680fd4eb4ca5e95d74d70', 'tests/test_complete_run.py': '5db381d51203123270a93b928e0e08908313045866ebca5cab8af55d60ce9d09'}
for name, expected in COMPLETE_SOURCE_HASHES.items():
    if file_sha256(CODE_DIR / name) != expected:
        raise ValueError(f'Complete-run source changed: {name}')
    target = extension_snapshot / name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CODE_DIR / name, target)
print('Complete-run sources verified.')


In [ ]:
#@title Run the data audit
import pandas as pd
from IPython.display import display

AUDIT_DIR = PROJECT_DIR / "data_audit"
run_python([CODE_DIR / "nested_loso.py", "--data", DATA_ZIP,
            "--mode", "audit", "--output", AUDIT_DIR], "data_audit")
AUDIT = json.loads((AUDIT_DIR / "audit" / "dataset_audit.json").read_text())
TASK_INFO = {row["task"]: row for row in AUDIT["tasks"]}
display(pd.DataFrame(AUDIT["tasks"])[["task", "n_participants", "n_cycles", "timepoints",
                                      "missing_input_values", "missing_target_values"]])
print("All nested split records:", AUDIT_DIR / "audit" / "nested_splits.json")


## 5 External workbook and the required source mapping

The embedded workbook below is your exact uploaded file, checked by SHA256.
**Its first row is numeric data, not a header.** All 101 samples are read.

The generated `external_column_map.csv` has 46 rows: one per left/right source
column. `cycle_id` is only a traceable source-column label. Fill in the real
`participant_id`, `trial_id`, `visit` (`baseline` or `post_intervention`), `include`
(`true`/`false`), and exclusion reasons. Repeated cycles from one person must use
the same participant ID. Do not invent IDs merely to make every column independent.
Complete body mass if the force values are in N, plus available age, sex,
medication state, HY stage, and intervention group.

In `external_import_settings.json`, verify:

- ankle units and the X/Y/Z axis that corresponds to each internal task;
- input/force signs from the acquisition conventions, never from prediction accuracy;
- force units (`N` or `N/kg`); N requires each participant's actual mass;
- identical column pairing across sheets, identity/visit mapping, disjoint cohorts;
- that the 101 rows represent 0–100% of the same complete gait-cycle definition.

After confirmation, the importer linearly interpolates **101 → 100 normalized
phase samples while retaining both endpoints**, converts physical units/mass, and
builds all six task bundles. It never estimates body mass from force amplitudes,
tunes signs using external errors, drops the first sample, or fills missing values
with zeros.

The supplied workbook cannot establish source exclusions or acquired trials absent
from its columns. Those counts are labelled unavailable; the generated flow tables
cover the supplied workbook. For complete source-cohort accounting, enrich the
generated `participants.csv`, `trials.csv`, and `cycles.csv` from acquisition records.

You can edit the metadata files in Drive while internal training runs. When ready,
rerun the study cell: completed internal work resumes and external evaluation runs
only after final-model freeze. The previously examined external cohort remains
labelled `previously_evaluated`.


In [ ]:
#@title Restore the supplied external workbook and make mapping templates
import base64
from external_workbook import make_mapping_templates, readiness
EXTERNAL_WORKBOOK = PROJECT_DIR / 'datasets' / 'Viglialoro2025_External_Validation_PD.xlsx'
EXTERNAL_WORKBOOK.parent.mkdir(parents=True, exist_ok=True)
embedded_external = """..."""
external_bytes = base64.b64decode(embedded_external)
if hashlib.sha256(external_bytes).hexdigest() != '21707b071bb8d1232138ee234c1980294f439451bc31ed7615f5ac8a6dec4659':
    raise ValueError('Embedded external workbook checksum mismatch')
if not EXTERNAL_WORKBOOK.exists():
    EXTERNAL_WORKBOOK.write_bytes(external_bytes)
elif file_sha256(EXTERNAL_WORKBOOK) != '21707b071bb8d1232138ee234c1980294f439451bc31ed7615f5ac8a6dec4659':
    raise ValueError('A different external workbook occupies the intended path; use a separate study folder')
del external_bytes, embedded_external
EXTERNAL_METADATA_DIR = PROJECT_DIR / 'external_source_metadata'
external_audit = make_mapping_templates(EXTERNAL_WORKBOOK, EXTERNAL_METADATA_DIR)
display(external_audit)
missing_external_metadata = readiness(EXTERNAL_WORKBOOK, EXTERNAL_METADATA_DIR)
print('External source metadata:', EXTERNAL_METADATA_DIR)
print('External readiness:', 'READY' if not missing_external_metadata else 'BLOCKED — complete the following source fields')
for reason in missing_external_metadata:
    print(' -', reason)


In [ ]:
#@title Download the external mapping template for editing
DOWNLOAD_MAPPING_TEMPLATE = True #@param {type:"boolean"}
from zipfile import ZIP_DEFLATED
if DOWNLOAD_MAPPING_TEMPLATE:
    template_zip = Path('/content/external_source_metadata.zip')
    with ZipFile(template_zip, 'w', compression=ZIP_DEFLATED) as archive:
        for p in EXTERNAL_METADATA_DIR.iterdir():
            if p.is_file():
                archive.write(p, p.name)
    files.download(str(template_zip))
print('Save completed CSV/JSON back to:', EXTERNAL_METADATA_DIR)


## 6 Complete-study settings and preflight

Defaults select **all six tasks, every outer participant, all seven families,
original deep search spaces, 10 candidates per deep family, up to 1,000 epochs,
and seed 42**. Early stopping uses held-out inner participants only.

All six coordinate modes are requested. With the current internal ZIP, only
`absolute` and `translation` are available. Pelvis/direction/body modes and physical
perturbations are recorded as blocked, not silently replaced. To enable them,
prepare real enriched internal bundles using the data contract at the end of this
notebook and start a separately named experiment with those inputs.

`PRIMARY_COORDINATES` is prespecified before external results. It controls the
one final deep model per component/side; other variants are internal sensitivity
analyses. No best seed, sign, ensemble weight, or coordinate mode is selected using
external errors. The optional LOSO ensemble uses equal waveform weights before
participant metrics; it is disabled by default.

The complete run uses a new experiment folder and preserves your smoke-test results.
Changing scientific settings after starting requires another experiment name.


In [ ]:
#@title Complete-study configuration
COMPLETE_EXPERIMENT = "reviewer_complete_v1" #@param {type:"string"}
TASK_SELECTION = "all" #@param {type:"string"}
INTERNAL_SOURCE = "original_zip" #@param ["original_zip", "enriched_bundles"]
INTERNAL_BUNDLE_ROOT = "/content/drive/MyDrive/Shida_Nested_LOSO/internal_cohorts" #@param {type:"string"}
REQUESTED_COORDINATES = "absolute,translation,pelvis,aligned,body,combined" #@param {type:"string"}
PRIMARY_COORDINATES = "absolute" #@param ["absolute", "translation", "pelvis", "aligned", "body", "combined"]
STUDY_SEED = 42 #@param {type:"integer"}
DEEP_CANDIDATES = 10 #@param {type:"integer"}
MAXIMUM_EPOCHS = 1000 #@param {type:"integer"}
EARLY_STOPPING_PATIENCE = 5 #@param {type:"integer"}
TRAINING_BATCH_SIZE = 32 #@param {type:"integer"}
SEARCH_PROFILE = "original" #@param ["original", "compact"]
STANDARDIZE_DEEP_INPUT = False #@param {type:"boolean"}
EXTERNAL_VISITS = "baseline" #@param ["baseline", "post_intervention", "baseline,post_intervention"]
INCLUDE_LOSO_ENSEMBLE = False #@param {type:"boolean"}

from data import TASK_ORDER
from revision_study import default_config
from complete_study import inspect_plan
if not re.fullmatch(r'[A-Za-z0-9_-]+', COMPLETE_EXPERIMENT):
    raise ValueError('Use letters, digits, underscore, or hyphen in the experiment name')
study_tasks = list(TASK_ORDER) if TASK_SELECTION.strip().lower() == 'all' else [x.strip() for x in TASK_SELECTION.split(',')]
study_cfg = default_config()
study_cfg.update(seed=STUDY_SEED, max_trials=DEEP_CANDIDATES, epochs=MAXIMUM_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE, batch_size=TRAINING_BATCH_SIZE,
    search_profile=SEARCH_PROFILE, scale_input=STANDARDIZE_DEEP_INPUT,
    external_visits=EXTERNAL_VISITS.split(','), external_status='previously_evaluated',
    include_loso_ensemble=INCLUDE_LOSO_ENSEMBLE, run_perturbations=True)
COMPLETE_ROOT = PROJECT_DIR / 'complete_studies' / COMPLETE_EXPERIMENT
COMPLETE_ROOT.mkdir(parents=True, exist_ok=True)
complete_plan = dict(tasks=study_tasks,
    coordinate_modes=[x.strip() for x in REQUESTED_COORDINATES.split(',')],
    primary_coordinates=PRIMARY_COORDINATES, internal_source=INTERNAL_SOURCE,
    internal_zip=str(DATA_ZIP), internal_bundle_root=INTERNAL_BUNDLE_ROOT,
    training=study_cfg, output_root=str(COMPLETE_ROOT),
    external_workbook=str(EXTERNAL_WORKBOOK), external_metadata_dir=str(EXTERNAL_METADATA_DIR),
    external_bundle_root=str(PROJECT_DIR / 'external_cohorts'))
COMPLETE_PLAN_PATH = PROJECT_DIR / f'{COMPLETE_EXPERIMENT}_requested_plan.json'
COMPLETE_PLAN_PATH.write_text(json.dumps(complete_plan, indent=2))
preflight = inspect_plan(complete_plan)
preflight.to_csv(COMPLETE_ROOT / 'preflight_preview.csv', index=False)
display(preflight[['task', 'coordinate', 'stage', 'status', 'fit_upper_bound', 'reason']])
print(f"Model-fit upper bound for available data: {int(preflight.fit_upper_bound.sum()):,}")
print('Experiment:', COMPLETE_ROOT)
print('Full research training starts in the next cell. This is not a smoke test.')


## 7 Run the complete available analysis

This cell executes the full plan without manually selecting each task or fold.
Each task completes its available nested coordinate studies, coordinate comparison,
final full-cohort refit, then external evaluation if ready. Training failures stop
with an explicit status; ordinary missing-data blocks allow other available work
to continue. After a disconnect, rerun setup and this cell with unchanged settings.

Internal raw-data participant counts remain 20 on the left and 19 on the right.
The notebook cannot create the additional participants needed to match the
manuscript's stated 26. “Full cohort” means all available valid people per task.


In [ ]:
#@title Start or resume the complete study
run_python([CODE_DIR / 'complete_study.py', '--plan', COMPLETE_PLAN_PATH], 'complete_study')
print('Study execution returned. Inspect the completed/blocked status below.')


## 8 Progress and results

Only stages marked `completed` have run. `blocked` means the stated source inputs
are still needed. The cohort statistics resample participants; per-participant
intervals resample complete trials and remain unavailable when fewer than two real
trials are known. Baseline and post-intervention results remain separate.

Inspect `paired_baseline_comparisons.csv` for each coordinate experiment. Positive
baseline-minus-deep error differences favor the deep pipeline; improvement must
be demonstrated by the full results rather than assumed from waveform correlation.
Subgroup summaries are descriptive and suppressed for sparse/unknown strata.


In [ ]:
#@title Show progress and available study summaries
progress_path = COMPLETE_ROOT / 'progress.csv'
if progress_path.exists():
    display(pd.read_csv(progress_path))
else:
    print('Training has not created a progress record yet.')
summary_path = COMPLETE_ROOT / 'all_internal_cohort_summaries.csv'
if summary_path.exists():
    display(pd.read_csv(summary_path))
for task in study_tasks:
    results = COMPLETE_ROOT / task / PRIMARY_COORDINATES / 'external_evaluation'
    if (results / 'complete.json').exists():
        print('External results:', task)
        display(pd.read_csv(results / 'domain_shift_audit' / 'cohort_flow_by_visit.csv'))
        display(pd.read_csv(results / 'cohort_summary.csv'))


## 9 Download outputs

The reports download includes progress, configurations, predictions, selected
settings, participant/cohort errors, paired baseline/coordinate comparisons, and
external flow/strata when completed. The full-experiment option adds model weights
and detailed tuning histories. The Drive experiment remains the resume location.


In [ ]:
#@title Download the complete-study results
DOWNLOAD_CONTENT = "reports" #@param ["reports", "complete_experiment"]
bundle = Path('/content') / f'{COMPLETE_EXPERIMENT}_{DOWNLOAD_CONTENT}.zip'
with ZipFile(bundle, 'w', compression=ZIP_DEFLATED) as archive:
    for path in COMPLETE_ROOT.rglob('*'):
        if not path.is_file() or path.suffix == '.tmp':
            continue
        relative = path.relative_to(COMPLETE_ROOT)
        if DOWNLOAD_CONTENT == 'reports':
            if path.suffix in {'.keras', '.joblib'} or any(p in {'candidates', 'tuner'} or p.startswith('candidate_') for p in relative.parts):
                continue
            if path.name.startswith('history'):
                continue
        archive.write(path, (Path('experiment') / relative).as_posix())
    for name in [*EXPECTED_SOURCE_HASHES, *EXTENSION_HASHES, *COMPLETE_SOURCE_HASHES]:
        archive.write(CODE_DIR / name, (Path('source_code') / name).as_posix())
    for path in EXTERNAL_METADATA_DIR.iterdir():
        if path.is_file():
            archive.write(path, (Path('external_source_metadata') / path.name).as_posix())
    archive.write(COMPLETE_PLAN_PATH, 'requested_plan.json')
files.download(str(bundle))


## Additional data contract for pelvis, direction, body-scale, and enriched source metadata

The original ZIP is sufficient for absolute/translation runs and every baseline.
Do not invent pelvis positions, participant heights, directions, trial IDs, or
external exclusion counts from the single-coordinate input files.

An **enriched internal** or **external** bundle is a directory with these five files:

| File | Required content |
|---|---|
| `waveforms.npz` | `cycle_ids` as strings, plus `x` and `y`, each `[N,100,1]`; or the 3D arrays below |
| `cycles.csv` | `cycle_id, participant_id, trial_id, visit, side, include, exclusion_reason` for all candidate cycles, including excluded ones |
| `trials.csv` | `participant_id, trial_id, visit, side, include, exclusion_reason` for all acquired trials, including those with zero valid cycles |
| `participants.csv` | One row per participant in the full roster; IDs, optional global identity, age, sex, medication state, HY stage, intervention group |
| `cohort.json` | Task, role, units, coordinate definition, acquisition systems/rates, walking protocol, gait-cycle definition, harmonization and provenance declarations |

IDs must be real and traceable; trial uniqueness is `(participant,trial,visit,side)`.
Use `baseline`, `post_intervention`, or `internal` visit labels and `left`/`right`
side labels. NPZ rows must map exactly to the ledger's included cycles **before**
the analysis visit filter. Excluded cycles need no waveform. No missing values
are zero-filled and no waveform is silently padded or excluded. Counts describe
the ledger supplied; code cannot recover unrecorded source exclusions.

**3D bundle:** use `ankle_xyz` and `grf_xyz`, each `[N,100,3]`, in a verified common
right-handed laboratory axis order `[forward,left,up]`. Also supply
`direction_xyz [N,3]`, a nonzero horizontal walking-direction vector determined
from pelvis progression or acquisition geometry, not GRF labels. Pelvis modes
need synchronized `pelvis_xyz [N,100,3]`; body modes need positive
`body_scale_m [N]` and a fixed definition such as measured standing height.
Do not assume separate AP/ML/vertical spreadsheet columns are synchronized.

All 3D variants predict the **same gait-aligned force component**: force vectors
are converted into gait axes once for every variant. This keeps target definitions
constant when comparing input coordinate modes. Comparing these outputs with an
older lab-frame analysis requires checking that the target definitions agree.

Coordinate definitions:

- `absolute`: original laboratory ankle coordinate.
- `translation`: subtract ankle position at phase zero within each cycle.
- `pelvis`: subtract the synchronized pelvis position at each phase.
- `aligned`: first-frame translation removal, then rotation into gait axes.
- `body`: first-frame translation removal, then divide by measured body scale.
- `combined`: pelvis centering, gait-axis alignment, then body-scale division.

Deterministic within-cycle transformations use only input kinematics; learned
standardizers are fitted on training participants only. External m/mm conversion
uses declared physical units, never external distribution fitting. GRFs must
already be N/kg. Verify filtering, event definition, interpolation to 100 phases,
axis signs, side and units before setting `harmonization_verified` to true.

The template starts with missing fields rather than invented cohort metadata.
If medication state or other characteristics vary by visit, create a separate
visit-specific bundle/metadata mapping before freezing that analysis; the provided
participant table assumes those covariates are constant within the analyzed visits.
